# ResNet-50 as the student (replaces the custom MicroBrainNet)

Everything else in the pipeline is unchanged: same teacher (MobileNetV3-Large), IM-only DA, gated pseudo-labels, logit+feature KD, background randomization, AdaBN, and the same Part 10 quantization/TFLite/op-check flow. **Changes vs the original notebook:** (1) Part 0: `STUDENT_ARCH='resnet50'`, `IMG_SIZE=224`, `LR=3e-4`, output dir renamed, multi-seed off; (2) new cell *Part 2-R* defines `ResNet50Student`/`build_student()`; (3) Parts 5C/0B use `build_student()`; (4) Part 7 multi-seed cells removed; (5) Grad-CAM target layer -> last ResNet bottleneck; (6) Part 10B rebuilt for ResNet-50 (native Keras + weight copy + TFLite FP32/INT8). All old outputs were cleared. Run top to bottom.

---
# Part 0: Imports & Configuration

In [ ]:
# ============================================================
# PART 0: IMPORTS AND CONFIGURATION
# ============================================================
import os, sys, time, random, warnings, copy, json, glob, math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report, roc_curve, auc)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR  = "/kaggle/working/resnet50_student"
FIGURES_DIR = f"{OUTPUT_DIR}/figures"
MODELS_DIR  = f"{OUTPUT_DIR}/models"
TABLES_DIR  = f"{OUTPUT_DIR}/tables"
for d in [FIGURES_DIR, MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------------
# FIX #8 -- the v6 student used 42,607 params = 41.6 KB INT8, i.e. 8% of the
# stated <500 KB budget. 96px throws away the fine texture that separates
# Tomato Septoria (many small spots) from Potato Early Blight (concentric rings).
# 128px + wider channels lands around 230 KB INT8 -- still comfortably inside budget,
# and ~250-400 ms/inference on an ESP32-S3 with PSRAM.
# ------------------------------------------------------------------
STUDENT_ARCH       = 'resnet50'   # <<< ResNet-50 replaces the custom MicroBrainNet as the STUDENT. Teacher unchanged.
STUDENT_PRETRAINED = True         # ImageNet init for ResNet-50 (the custom net was trained from scratch). Set False for a from-scratch run.
IMG_SIZE          = 224          # student view is now 224px (was 128) because ResNet-50 is used at its native resolution
TEACHER_IMG_SIZE  = 224
NUM_CLASSES       = 3
STUDENT_CHANNELS  = (32, 48, 64, 96, 96)     # v6 was (16, 24, 32, 48, 48)

BATCH_SIZE        = 48    # source
TARGET_BATCH_SIZE = 32    # FIX #7: smaller batch + drop_last=False keeps all 364 target images
TEACHER_BATCH_SIZE= 24

CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]

# ---------------- stage switches ----------------
RUN_SANITY_GATE        = True    # Part 0B: source-only run. DO NOT SKIP on the first execution.
USE_BG_RANDOMIZATION   = True    # FIX #6
USE_TEACHER_KD         = True
TEACHER_USE_PSEUDO     = False   # FIX #1: OFF. This is what poisoned v6. Turn on only after
                                 #         the teacher's clean baseline is known.
STUDENT_USE_PSEUDO     = True    # gated + auto-disabling (see Part 3/4)
RUN_MULTI_SEED         = False    # Part 7 (expensive: re-runs the student per seed) -- turned ON:
                                  # a single-seed number is not defensible on a 79-image test set
SEEDS                  = [42, 1337, 2024]

# ---------------- teacher ----------------
TEACHER_BACKBONE   = 'mobilenet_v3_large'
TEACHER_EPOCHS     = 80
TEACHER_LR_BACKBONE= 3e-4        # FIX #10: discriminative LR instead of a cosmetic block freeze
TEACHER_LR_HEAD    = 3e-3
TEACHER_WARMUP     = 3
TEACHER_PATIENCE   = 30

# ---------------- student ----------------
EPOCHS         = 120
LR             = 3e-4 if STUDENT_ARCH == 'resnet50' else 1.5e-3   # 1.5e-3 AdamW would wreck ImageNet-pretrained ResNet-50 weights
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTHING= 0.1
WARMUP_EPOCHS  = 5
PATIENCE       = 40

# ---------------- domain adaptation ----------------
# EXPERIMENT: IM-only. v7 ran four alignment losses together (CORAL + MMD + IM + MCC).
# This run switches all four off except Information Maximization to see how much of the
# adaptation the plain IM term can carry on its own -- CORAL/MMD/MCC stay fully implemented
# below (Part 3) and can be flipped back on by setting these four switches to True.
USE_CORAL = False
USE_MMD   = False
USE_IM    = True
USE_MCC   = False

# FIX #9: CORAL down from 5.0. In v6 CORAL fell to 1e-4 while target accuracy never moved --
# the classic signature of degenerate alignment (domains matched by destroying class structure).
LAMBDA_CORAL_MAX  = 0.5
LAMBDA_MMD_MAX    = 0.5
LAMBDA_IM_MAX     = 0.5    # FIX #3: Information Maximization replaces bare entropy-min
LAMBDA_MCC_MAX    = 1.0    # FIX #3: Minimum Class Confusion -- class-aware, stable
LAMBDA_PSEUDO_MAX = 0.3

# FIX #2: v6 iterated the source loader (41 batches) and cycled the 5-batch target loader
# 8.2x per epoch, so 364 target images dominated every alignment term. Cap it.
DA_TARGET_PASSES  = 2

# ---------------- pseudo-labeling (FIX #1) ----------------
PSEUDO_CONF            = 0.95
PSEUDO_TOPK_PER_CLASS  = 8      # class-balanced instead of a global threshold
PSEUDO_MIN_MONITOR_ACC = 0.85   # hard safety valve: below this, self-training switches itself off
PSEUDO_PLATEAU_EPOCHS  = 8      # only start once target-val has stopped improving
EMA_DECAY              = 0.999

# ---------------- knowledge distillation (FIX #4) ----------------
LAMBDA_KD      = 1.0    # constant, NOT ramped, and no longer 3x the CE weight
LAMBDA_FEATKD  = 1.0
KD_TEMPERATURE = 4.0

SMOOTH_WINDOW = 5

CFG = dict(coral=LAMBDA_CORAL_MAX, mmd=LAMBDA_MMD_MAX, im=LAMBDA_IM_MAX, mcc=LAMBDA_MCC_MAX,
           pseudo=LAMBDA_PSEUDO_MAX, kd=LAMBDA_KD, featkd=LAMBDA_FEATKD, T=KD_TEMPERATURE,
           da_passes=DA_TARGET_PASSES, pseudo_conf=PSEUDO_CONF,
           pseudo_topk=PSEUDO_TOPK_PER_CLASS, pseudo_min_acc=PSEUDO_MIN_MONITOR_ACC)

print(f"\nIMG={IMG_SIZE} channels={STUDENT_CHANNELS} BS={BATCH_SIZE}/{TARGET_BATCH_SIZE}")
print(f"DA technique switches: CORAL={USE_CORAL} MMD={USE_MMD} IM={USE_IM} MCC={USE_MCC}")
print(f"DA lambdas (only active ones apply): coral={LAMBDA_CORAL_MAX} mmd={LAMBDA_MMD_MAX} "
      f"IM={LAMBDA_IM_MAX} MCC={LAMBDA_MCC_MAX}")
print(f"KD: lambda={LAMBDA_KD} (constant) T={KD_TEMPERATURE} feat={LAMBDA_FEATKD}")
print(f"Pseudo: teacher={TEACHER_USE_PSEUDO} student={STUDENT_USE_PSEUDO} "
      f"conf>={PSEUDO_CONF} topk={PSEUDO_TOPK_PER_CLASS}/class auto-off<{PSEUDO_MIN_MONITOR_ACC}")

Device: cuda | PyTorch: 2.10.0+cu128
GPU: Tesla T4

IMG=224 channels=(32, 48, 64, 96, 96) BS=48/32
DA technique switches: CORAL=False MMD=False IM=True MCC=False
DA lambdas (only active ones apply): coral=0.5 mmd=0.5 IM=0.5 MCC=1.0
KD: lambda=1.0 (constant) T=4.0 feat=1.0
Pseudo: teacher=False student=True conf>=0.95 topk=8/class auto-off<0.85


---
# Part 1: Data — indexing, splits, domain-gap augmentation

Three things happen here, in order:

1. **Indexing** — walk both dataset roots, match folder-name aliases per domain, build
   `img_paths` / `labels`. Then one stratified 70/15/15 split per domain, computed once and
   reused by every model so teacher and student are always evaluated on identical images.
2. **Background randomization (FIX #6)** — PlantVillage is a centred leaf on a uniform grey
   backdrop; PlantDoc is a leaf somewhere in a cluttered field. A model trained on the former
   learns "uniform background => confident prediction" and, as published Grad-CAM analyses of
   this exact shift show, moves its attention onto the background at test time. So: recover the
   leaf mask (from PlantVillage's own `segmented/` split where available, otherwise a
   border-colour heuristic) and composite the leaf onto random crops of **PlantDoc train**
   images. Target val/test images are never used as backgrounds.
3. **`DualResDataset` (FIX #4)** — the reason v6's distillation failed. v6 augmented at 96px and
   then bilinearly upsampled to 224px for the teacher, so the teacher — fine-tuned on real 224px
   photos — was scored on blurry stretched inputs and returned near-useless soft targets, which
   were then weighted 3x the classification loss. Here a single geometric+photometric
   augmentation is applied once at 256px, then rendered twice: 128px/[-1,1] for the student and
   224px/ImageNet-norm for the teacher. Same crop, same jitter, same pixels, correct scale for both.

Every sample is a 5-tuple `(x_student, x_teacher, x_weak, label, domain)`. `x_weak` is a
resize+flip-only view used solely to *generate* pseudo-labels, while the model trains on the
strongly augmented `x_student` — the weak/strong split that makes FixMatch-style self-training
stable. Unused views are returned as a 1-element placeholder so collation stays uniform.

In [ ]:
# ============================================================
# PART 1A: DATASET INDEX
# ============================================================
SOURCE_ALL    = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
SEGMENTED_ALL = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented"
TARGET_ALL    = "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset"

SOURCE_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato___Septoria_leaf_spot"],
    "Potato_Early_Blight":       ["Potato___Early_blight"],
    "Corn_Leaf_Blight":          ["Corn_(maize)___Northern_Leaf_Blight"],
}
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


class LeafIndex:
    """Path/label index only -- no transforms. Walks root_dir recursively and pools
    every image under any folder whose normalized name matches an alias for a class."""

    def __init__(self, root_dir, classes, class_dir_map, domain_label=0, name=""):
        self.img_paths, self.labels = [], []
        self.classes, self.domain_label = classes, domain_label

        def _norm(s):
            return s.strip().lower().replace(" ", "").replace("_", "")

        for idx, cls in enumerate(classes):
            alias = {_norm(a) for a in class_dir_map.get(cls, [cls])}
            matched = [dp for dp, _, _ in os.walk(root_dir) if _norm(os.path.basename(dp)) in alias]
            for d in matched:
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.img_paths.append(os.path.join(d, f)); self.labels.append(idx)
            if not matched:
                print(f"    WARNING: no folder matched '{cls}' under {root_dir}")

        print(f"  [{name}] {len(self.img_paths)} images")
        for i, c in enumerate(classes):
            print(f"    {c}: {self.labels.count(i)}")

    def subset(self, indices):
        s = copy.copy(self)
        s.img_paths = [self.img_paths[i] for i in indices]
        s.labels    = [self.labels[i] for i in indices]
        return s


def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx_all = list(range(len(labels)))
    idx_tv, idx_test = train_test_split(idx_all, test_size=test_frac, stratify=labels,
                                        random_state=seed)
    lab_tv = [labels[i] for i in idx_tv]
    idx_tr, idx_val = train_test_split(idx_tv, test_size=val_frac / (1 - test_frac),
                                       stratify=lab_tv, random_state=seed)
    return idx_tr, idx_val, idx_test


for nm, p in [('PlantVillage color', SOURCE_ALL), ('PlantVillage segmented', SEGMENTED_ALL),
              ('PlantDoc', TARGET_ALL)]:
    print(f"  {nm:24s}: {'OK ' if os.path.isdir(p) else '!! NOT FOUND !!'} {p}")

print("\n--- Source (PlantVillage) ---")
src_index = LeafIndex(SOURCE_ALL, CLASSES, SOURCE_CLASS_DIRS, 0, "Source")
print("\n--- Target (PlantDoc) ---")
tgt_index = LeafIndex(TARGET_ALL, CLASSES, TARGET_CLASS_DIRS, 1, "Target")

src_tr_i, src_va_i, src_te_i = stratified_split_indices(src_index.labels)
tgt_tr_i, tgt_va_i, tgt_te_i = stratified_split_indices(tgt_index.labels)

src_train_idx = src_index.subset(src_tr_i)
src_val_idx   = src_index.subset(src_va_i)
src_test_idx  = src_index.subset(src_te_i)
tgt_train_idx = tgt_index.subset(tgt_tr_i)
tgt_val_idx   = tgt_index.subset(tgt_va_i)
tgt_test_idx  = tgt_index.subset(tgt_te_i)

print(f"\nSource  train/val/test: {len(src_train_idx.labels)}/"
      f"{len(src_val_idx.labels)}/{len(src_test_idx.labels)}")
print(f"Target  train/val/test: {len(tgt_train_idx.labels)}/"
      f"{len(tgt_val_idx.labels)}/{len(tgt_test_idx.labels)}")
print(f"\nNOTE: target test = {len(tgt_test_idx.labels)} images -> 1 image = "
      f"{100/len(tgt_test_idx.labels):.2f} accuracy points. Never quote a single point estimate.")

  PlantVillage color      : OK  /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
  PlantVillage segmented  : OK  /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/segmented
  PlantDoc                : OK  /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset

--- Source (PlantVillage) ---
  [Source] 3756 images
    Tomato_Septoria_Leaf_Spot: 1771
    Potato_Early_Blight: 1000
    Corn_Leaf_Blight: 985

--- Target (PlantDoc) ---
  [Target] 522 images
    Tomato_Septoria_Leaf_Spot: 157
    Potato_Early_Blight: 171
    Corn_Leaf_Blight: 194

Source  train/val/test: 2628/564/564
Target  train/val/test: 364/79/79

NOTE: target test = 79 images -> 1 image = 1.27 accuracy points. Never quote a single point estimate.


In [ ]:
# ============================================================
# VIZ 1A-1: Raw source images — one per class (individual figures)
# ============================================================
for ci, cls in enumerate(CLASSES):
    idxs = [j for j, l in enumerate(src_index.labels) if l == ci]
    if idxs:
        img = Image.open(src_index.img_paths[idxs[0]]).convert("RGB")
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.set_title(f"Source (PlantVillage): {cls}", fontsize=12, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/raw_source_{cls}.png", dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved: raw_source_{cls}.png")

  Saved: raw_source_Tomato_Septoria_Leaf_Spot.png
  Saved: raw_source_Potato_Early_Blight.png
  Saved: raw_source_Corn_Leaf_Blight.png


In [ ]:
# ============================================================
# VIZ 1A-2: Raw target images — one per class (individual figures)
# ============================================================
for ci, cls in enumerate(CLASSES):
    idxs = [j for j, l in enumerate(tgt_index.labels) if l == ci]
    if idxs:
        img = Image.open(tgt_index.img_paths[idxs[0]]).convert("RGB")
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.set_title(f"Target (PlantDoc): {cls}", fontsize=12, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/raw_target_{cls}.png", dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved: raw_target_{cls}.png")

  Saved: raw_target_Tomato_Septoria_Leaf_Spot.png
  Saved: raw_target_Potato_Early_Blight.png
  Saved: raw_target_Corn_Leaf_Blight.png


In [ ]:
# ============================================================
# VIZ 1A-3: Dataset size comparison bar chart
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(NUM_CLASSES)
w = 0.35
src_counts = [src_index.labels.count(i) for i in range(NUM_CLASSES)]
tgt_counts = [tgt_index.labels.count(i) for i in range(NUM_CLASSES)]
bars1 = ax.bar(x - w/2, src_counts, w, label='Source (PlantVillage)', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + w/2, tgt_counts, w, label='Target (PlantDoc)', color='#FF5722', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([c[:20] for c in CLASSES], rotation=15, fontsize=9)
ax.set_ylabel('Number of Images')
ax.set_title('Dataset Size: Source vs Target per Class', fontweight='bold')
ax.legend()
for b in bars1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), str(int(b.get_height())),
            ha='center', va='bottom', fontsize=9)
for b in bars2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), str(int(b.get_height())),
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/dataset_size_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 1A-4: Split sizes (individual bar chart)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
splits = ['Train', 'Val', 'Test']
src_split = [len(src_train_idx.labels), len(src_val_idx.labels), len(src_test_idx.labels)]
tgt_split = [len(tgt_train_idx.labels), len(tgt_val_idx.labels), len(tgt_test_idx.labels)]
x = np.arange(len(splits))
w = 0.35
b1 = ax.bar(x - w/2, src_split, w, label='Source', color='#2196F3', alpha=0.85)
b2 = ax.bar(x + w/2, tgt_split, w, label='Target', color='#FF5722', alpha=0.85)
for b in b1:
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), str(int(b.get_height())),
            ha='center', va='bottom', fontsize=10)
for b in b2:
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), str(int(b.get_height())),
            ha='center', va='bottom', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel('Number of Images')
ax.set_title('Train/Val/Test Split Sizes', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/split_sizes.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PART 1B: BACKGROUND RANDOMIZATION  (FIX #6)
# ============================================================
# PlantVillage: centred leaf, uniform grey backdrop. PlantDoc: leaf in a cluttered field.
# A source-only model latches onto the backdrop as a shortcut. Cutting the leaf out and
# pasting it onto real field backgrounds removes that shortcut at the data level, which is
# more effective than any loss-side alignment term.
#
# Mask source, in order of preference:
#   1. PlantVillage's own segmented/ split (exact, free) -- matched by filename stem
#   2. Border-colour heuristic + morphological cleanup (works because the backdrop is uniform)
# Backgrounds: random crops of PlantDoc TRAIN images only (never val/test -> no leakage).

from scipy import ndimage as ndi


def build_segmented_lookup(seg_root, class_dir_map, classes):
    """stem -> segmented path. PlantVillage segmented filenames carry a '_final_masked'
    suffix, so match on the leading UUID-ish stem rather than the full filename."""
    lut = {}
    if not (seg_root and os.path.isdir(seg_root)):
        return lut

    def _norm(s):
        return s.strip().lower().replace(" ", "").replace("_", "")

    alias = set()
    for c in classes:
        alias |= {_norm(a) for a in class_dir_map.get(c, [c])}
    for dp, _, files in os.walk(seg_root):
        if _norm(os.path.basename(dp)) not in alias:
            continue
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                stem = os.path.splitext(f)[0].replace('_final_masked', '')
                lut[stem] = os.path.join(dp, f)
    return lut


SEG_LUT = build_segmented_lookup(SEGMENTED_ALL, SOURCE_CLASS_DIRS, CLASSES) if USE_BG_RANDOMIZATION else {}
print(f"Segmented-mask lookup: {len(SEG_LUT)} entries "
      f"({'exact masks available' if SEG_LUT else 'falling back to the border heuristic'})")


def estimate_leaf_mask(img, seg_path=None):
    """Boolean HxW mask, True = leaf. Returns None when the result looks untrustworthy,
    in which case the caller simply skips background randomization for that image."""
    if seg_path is not None and os.path.exists(seg_path):
        try:
            seg = np.asarray(Image.open(seg_path).convert("RGB").resize(img.size), np.float32)
            m = seg.sum(2) > 24          # segmented split paints the background pure black
            if 0.05 < m.mean() < 0.97:
                return ndi.binary_closing(m, np.ones((5, 5)))
        except Exception:
            pass

    a = np.asarray(img.convert("RGB"), np.float32)
    b = np.concatenate([a[:6].reshape(-1, 3), a[-6:].reshape(-1, 3),
                        a[:, :6].reshape(-1, 3), a[:, -6:].reshape(-1, 3)])
    med = np.median(b, 0)
    d = np.linalg.norm(a - med, axis=2)
    m = d > max(35.0, np.percentile(d, 40))
    m = ndi.binary_opening(m, np.ones((3, 3)))
    m = ndi.binary_closing(m, np.ones((9, 9)))
    m = ndi.binary_fill_holes(m)

    lab, n = ndi.label(m)                # keep only the largest blob (the leaf)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (int(np.argmax(sizes)) + 1)
    if not (0.05 < m.mean() < 0.95):
        return None
    return m


def build_background_bank(index_obj, n=180, size=320):
    """Field backgrounds taken from PlantDoc TRAIN images only."""
    rng = np.random.RandomState(SEED)
    paths = list(index_obj.img_paths)
    rng.shuffle(paths)
    bank = []
    for p in paths[:n]:
        try:
            bank.append(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))
        except Exception:
            continue
    return bank


BG_BANK = build_background_bank(tgt_train_idx) if USE_BG_RANDOMIZATION else []
print(f"Background bank: {len(BG_BANK)} field images (PlantDoc TRAIN split only)")


MASK_SIZE = 192


def precompute_masks(index_obj):
    """Masks are computed ONCE here and stored bit-packed (~6 KB each), instead of being
    recomputed inside the dataloader on every __getitem__. Without this the scipy
    morphology becomes the training bottleneck."""
    if not USE_BG_RANDOMIZATION:
        return {}
    cache, t0 = {}, time.time()
    for n, p in enumerate(index_obj.img_paths):
        try:
            im = Image.open(p).convert("RGB").resize((MASK_SIZE, MASK_SIZE), Image.BILINEAR)
            stem = os.path.splitext(os.path.basename(p))[0]
            m = estimate_leaf_mask(im, SEG_LUT.get(stem))
            if m is not None:
                cache[p] = np.packbits(m)
        except Exception:
            pass
        if (n + 1) % 500 == 0:
            print(f"    masks {n+1}/{len(index_obj.img_paths)} ({time.time()-t0:.0f}s)")
    print(f"  usable leaf masks: {len(cache)}/{len(index_obj.img_paths)} "
          f"in {time.time()-t0:.0f}s")
    return cache


MASK_CACHE = precompute_masks(src_train_idx) if USE_BG_RANDOMIZATION else {}


def randomize_background(img, path, rng):
    """Composite the leaf onto a random field background with a soft alpha edge."""
    if not BG_BANK or path not in MASK_CACHE:
        return img
    m = np.unpackbits(MASK_CACHE[path])[:MASK_SIZE * MASK_SIZE].reshape(MASK_SIZE, MASK_SIZE)

    bg = BG_BANK[rng.randint(len(BG_BANK))]
    W, H = img.size
    bw, bh = bg.size
    cw = rng.randint(int(bw * 0.4), bw + 1)
    x0 = rng.randint(0, bw - cw + 1); y0 = rng.randint(0, bh - cw + 1)
    bg = bg.crop((x0, y0, x0 + cw, y0 + cw)).resize((W, H), Image.BILINEAR)

    alpha = (Image.fromarray((m * 255).astype(np.uint8))
             .resize((W, H), Image.BILINEAR).filter(ImageFilter.GaussianBlur(1.5)))
    out = Image.composite(img, bg, alpha)
    if rng.rand() < 0.3:                 # sometimes blur the background only
        out = Image.composite(out, bg.filter(ImageFilter.GaussianBlur(2.0)), alpha)
    return out

Segmented-mask lookup: 3756 entries (exact masks available)
Background bank: 180 field images (PlantDoc TRAIN split only)
    masks 500/2628 (7s)
    masks 1000/2628 (13s)
    masks 1500/2628 (20s)
    masks 2000/2628 (27s)
    masks 2500/2628 (34s)
  usable leaf masks: 2628/2628 in 36s


In [ ]:
# ============================================================
# VIZ 1B-1: Leaf mask visualization — individual images
# ============================================================
if USE_BG_RANDOMIZATION and MASK_CACHE:
    sample_paths = list(MASK_CACHE.keys())[:3]
    for si, p in enumerate(sample_paths):
        # Original image
        img = Image.open(p).convert("RGB").resize((MASK_SIZE, MASK_SIZE), Image.BILINEAR)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.set_title(f"Original Source Image {si+1}", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/mask_original_{si}.png", dpi=150, bbox_inches='tight')
        plt.show()

        # Mask
        m = np.unpackbits(MASK_CACHE[p])[:MASK_SIZE*MASK_SIZE].reshape(MASK_SIZE, MASK_SIZE)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(m, cmap='gray')
        ax.set_title(f"Leaf Mask {si+1} (white=leaf, black=background)", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/mask_binary_{si}.png", dpi=150, bbox_inches='tight')
        plt.show()

        # Masked leaf
        img_arr = np.asarray(img)
        masked = img_arr * m[:,:,None]
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(masked.astype(np.uint8))
        ax.set_title(f"Masked Leaf {si+1} (background removed)", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/mask_applied_{si}.png", dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("Background randomization disabled or no masks available.")

In [ ]:
# ============================================================
# VIZ 1B-2: Background randomization before/after (individual)
# ============================================================
if USE_BG_RANDOMIZATION and BG_BANK and MASK_CACHE:
    sample_paths = list(MASK_CACHE.keys())[:3]
    rng = np.random.RandomState(SEED)
    for si, p in enumerate(sample_paths):
        # Before (original)
        orig = Image.open(p).convert("RGB").resize((256, 256), Image.BILINEAR)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(orig)
        ax.set_title(f"Before BG Randomization {si+1}", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/bg_before_{si}.png", dpi=150, bbox_inches='tight')
        plt.show()

        # After (composited onto field background)
        composite = randomize_background(orig, p, rng)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(composite)
        ax.set_title(f"After BG Randomization {si+1}", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/bg_after_{si}.png", dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("Background randomization disabled.")

In [ ]:
# ============================================================
# VIZ 1B-3: Background bank samples (individual)
# ============================================================
if BG_BANK:
    for bi in range(min(3, len(BG_BANK))):
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(BG_BANK[bi])
        ax.set_title(f"Background Image {bi+1} (from PlantDoc TRAIN)", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/bg_sample_{bi}.png", dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No background bank available.")

In [ ]:
# ============================================================
# PART 1C: DualResDataset  (FIX #4 + FIX #6)
# ============================================================
# One augmentation, rendered at both resolutions. This is the cell that repairs distillation:
# v6 augmented at 96px and bilinearly upsampled to 224px for the teacher, so the teacher saw
# blurry stretched inputs it was never fine-tuned on and emitted soft targets that were then
# weighted 3x the CE loss. Here both views come from the same 256px augmented image.
#
# Returns (x_student, x_teacher, x_weak_student, x_weak_teacher, label, domain).
# Two weak views are needed because pseudo-labels must be generated at whatever resolution the
# model being trained consumes -- 128px for the student, 224px for the teacher.
# Unrequested views are 1-element placeholders so the default collate still stacks cleanly.

_PLACEHOLDER = torch.zeros(1)


class DualResDataset(Dataset):
    def __init__(self, index_obj, train=True, emit_teacher=False, emit_weak=False,
                 bg_random_p=0.0, img_size=IMG_SIZE, teacher_size=TEACHER_IMG_SIZE):
        self.idx = index_obj
        self.train = train
        self.emit_teacher = emit_teacher
        self.emit_weak = emit_weak
        self.bg_p = bg_random_p if index_obj.domain_label == 0 else 0.0
        self.img_size, self.teacher_size = img_size, teacher_size

        # -- strong, domain-gap-aware augmentation (FIX #6) --
        # RandomResizedCrop(0.25-1.0) is the important one: it forces the model onto lesion
        # texture instead of whole-leaf silhouette, which does not survive the domain shift.
        self.geo_train = transforms.Compose([
            transforms.RandomResizedCrop(256, scale=(0.25, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.2),
            transforms.RandomApply([transforms.RandomRotation(25)], p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.1),
            transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.3),
        ])
        self.geo_eval = transforms.Resize((256, 256))
        self.geo_weak = transforms.Compose([transforms.Resize((256, 256)),
                                            transforms.RandomHorizontalFlip(0.5)])

        self.to_student = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)])
        self.to_teacher = transforms.Compose([
            transforms.Resize((teacher_size, teacher_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.erase = transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value='random')

    def __len__(self):
        return len(self.idx.img_paths)

    def __getitem__(self, i):
        path = self.idx.img_paths[i]
        img = Image.open(path).convert("RGB")
        y, d = self.idx.labels[i], self.idx.domain_label

        if self.train and self.bg_p > 0:
            rng = np.random.RandomState((i * 7919 + int(time.time() * 1e3)) % (2 ** 31))
            if rng.rand() < self.bg_p:
                img = randomize_background(img.resize((256, 256), Image.BILINEAR), path, rng)

        base = self.geo_train(img) if self.train else self.geo_eval(img)

        xs = self.to_student(base)
        if self.train:
            xs = self.erase(xs)
        xt = self.to_teacher(base) if self.emit_teacher else _PLACEHOLDER
        if self.emit_weak:
            wbase = self.geo_weak(img)
            xw_s = self.to_student(wbase)
            xw_t = self.to_teacher(wbase) if self.emit_teacher else _PLACEHOLDER
        else:
            xw_s = xw_t = _PLACEHOLDER
        return xs, xt, xw_s, xw_t, y, d

In [ ]:
# ============================================================
# PART 1D: DATASETS AND LOADERS
# ============================================================
NW = 4          # lower to 2 if the Kaggle worker runs out of RAM

def mk_loader(ds, bs, shuffle, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=NW,
                      pin_memory=True, drop_last=drop_last, persistent_workers=(NW > 0))

BG_P = 0.5 if USE_BG_RANDOMIZATION else 0.0

# -- train --
src_train_ds = DualResDataset(src_train_idx, True,  emit_teacher=True, bg_random_p=BG_P)
tgt_train_ds = DualResDataset(tgt_train_idx, True,  emit_teacher=True, emit_weak=True)

# -- eval (no augmentation anywhere) --
src_val_ds   = DualResDataset(src_val_idx,   False, emit_teacher=True)
src_test_ds  = DualResDataset(src_test_idx,  False, emit_teacher=True)
tgt_val_ds   = DualResDataset(tgt_val_idx,   False, emit_teacher=True)
tgt_test_ds  = DualResDataset(tgt_test_idx,  False, emit_teacher=True)

# FIX #12: v6 measured "source train accuracy" through the augmented training loader, which
# understated it and therefore corrupted the reported generalization gap. Clean loader:
src_train_eval_ds = DualResDataset(src_train_idx, False, emit_teacher=True)

src_train_loader = mk_loader(src_train_ds, BATCH_SIZE, True,  drop_last=True)
# FIX #7: drop_last=False -- v6 silently discarded 44 of 364 target images every epoch.
tgt_train_loader = mk_loader(tgt_train_ds, TARGET_BATCH_SIZE, True, drop_last=False)

src_val_loader        = mk_loader(src_val_ds,   BATCH_SIZE, False)
src_test_loader       = mk_loader(src_test_ds,  BATCH_SIZE, False)
tgt_val_loader        = mk_loader(tgt_val_ds,   BATCH_SIZE, False)
tgt_test_loader       = mk_loader(tgt_test_ds,  BATCH_SIZE, False)
src_train_eval_loader = mk_loader(src_train_eval_ds, BATCH_SIZE, False)

print(f"source batches/epoch : {len(src_train_loader)}")
print(f"target batches/epoch : {len(tgt_train_loader)}")
print(f"DA steps/epoch       : {min(len(src_train_loader), DA_TARGET_PASSES*len(tgt_train_loader))}"
      f"  -> each target image seen ~{DA_TARGET_PASSES}x/epoch")
print(f"  (v6 cycled the target loader {len(src_train_loader)/max(len(tgt_train_loader),1):.1f}x "
      f"per epoch -- that over-weighting is what drove CORAL to 1e-4 with no accuracy gain)")

xb = next(iter(src_train_loader))
print(f"\nbatch shapes: student={tuple(xb[0].shape)} teacher={tuple(xb[1].shape)}")
print(f"student range: [{xb[0].min():.2f}, {xb[0].max():.2f}]  (expect ~[-1, 1])")
xt_b = next(iter(tgt_train_loader))
print(f"target weak views: student={tuple(xt_b[2].shape)} teacher={tuple(xt_b[3].shape)}")

source batches/epoch : 54
target batches/epoch : 12
DA steps/epoch       : 24  -> each target image seen ~2x/epoch
  (v6 cycled the target loader 4.5x per epoch -- that over-weighting is what drove CORAL to 1e-4 with no accuracy gain)

batch shapes: student=(48, 3, 224, 224) teacher=(48, 3, 224, 224)
student range: [-4.01, 4.33]  (expect ~[-1, 1])
target weak views: student=(32, 3, 224, 224) teacher=(32, 3, 224, 224)


In [ ]:
# ============================================================
# VIZ 1D-1: Multiple augmentations of the same source image (individual)
# ============================================================
vis_ds_aug = DualResDataset(src_train_idx, True, bg_random_p=BG_P)
sample_idx = 0
for aug_i in range(6):
    x = vis_ds_aug[sample_idx][0]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(np.clip(x.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
    ax.set_title(f"Source Augmentation #{aug_i+1} (same base image)", fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/augmentation_variant_{aug_i}.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# VIZ 1D-2: Student (128px) vs Teacher (224px) views (individual)
# ============================================================
vis_ds_dual = DualResDataset(src_train_idx, True, emit_teacher=True, bg_random_p=BG_P)
for si in range(3):
    xs, xt, _, _, y, _ = vis_ds_dual[si * 10]
    # Student view
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(np.clip(xs.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
    ax.set_title(f"Student View (128px) — {CLASSES[y][:20]}", fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/student_view_{si}.png", dpi=150, bbox_inches='tight')
    plt.show()

    # Teacher view
    t_img = xt.permute(1, 2, 0).numpy()
    t_img = t_img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(np.clip(t_img, 0, 1))
    ax.set_title(f"Teacher View (224px) — {CLASSES[y][:20]}", fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_view_{si}.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# PART 1E: VISUAL CHECK -- is background randomization actually working?
# ============================================================
# Worth eyeballing before a long run. If the composites look wrong (leaf cut in half, mask
# inverted), set USE_BG_RANDOMIZATION=False rather than training on corrupted images.

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle("Row 1-2: source, augmented (background randomized) | Row 3: target (PlantDoc)",
             fontsize=14, fontweight='bold')

vis_ds = DualResDataset(src_train_idx, True, bg_random_p=1.0)
for k in range(6):
    for r in range(2):
        x = vis_ds[k * 3 + r][0]
        axes[r][k].imshow(np.clip(x.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
        axes[r][k].axis('off')

vis_t = DualResDataset(tgt_train_idx, False)
for k in range(6):
    x = vis_t[k * 5][0]
    axes[2][k].imshow(np.clip(x.permute(1, 2, 0).numpy() * 0.5 + 0.5, 0, 1))
    axes[2][k].axis('off')

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/augmentation_check.png", dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, idxo, t, c in [(axes[0], src_train_idx, 'Source train', '#2196F3'),
                       (axes[1], tgt_train_idx, 'Target train', '#FF5722')]:
    cnt = [idxo.labels.count(i) for i in range(NUM_CLASSES)]
    ax.bar(range(NUM_CLASSES), cnt, color=c, alpha=.85)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels([c2[:14] for c2 in CLASSES], rotation=15, fontsize=9)
    ax.set_title(t, fontweight='bold')
    for i, v in enumerate(cnt):
        ax.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

shares = np.array([src_index.labels.count(i) for i in range(NUM_CLASSES)], float)
shares /= shares.sum()
print("Source class shares:", {CLASSES[i]: round(float(shares[i]), 4) for i in range(NUM_CLASSES)})
print(f"\nCOLLAPSE REFERENCE -- if source val accuracy parks at one of these, a class has been")
print(f"erased from the feature space (this is exactly what v6's TA did at 0.730):")
for i in range(NUM_CLASSES):
    print(f"  all of {CLASSES[i][:22]:<22s} absorbed elsewhere -> {1 - shares[i]:.4f}")

Source class shares: {'Tomato_Septoria_Leaf_Spot': 0.4715, 'Potato_Early_Blight': 0.2662, 'Corn_Leaf_Blight': 0.2622}

COLLAPSE REFERENCE -- if source val accuracy parks at one of these, a class has been
erased from the feature space (this is exactly what v6's TA did at 0.730):
  all of Tomato_Septoria_Leaf_S absorbed elsewhere -> 0.5285
  all of Potato_Early_Blight    absorbed elsewhere -> 0.7338
  all of Corn_Leaf_Blight       absorbed elsewhere -> 0.7378


In [ ]:
# ============================================================
# VIZ 1E-1: Source class distribution (individual)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
cnt = [src_train_idx.labels.count(i) for i in range(NUM_CLASSES)]
bars = ax.bar(range(NUM_CLASSES), cnt, color='#2196F3', alpha=.85)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([c[:20] for c in CLASSES], rotation=15, fontsize=9)
ax.set_title('Source Train — Class Distribution', fontweight='bold')
ax.set_ylabel('Number of Images')
for i, v in enumerate(cnt):
    ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_dist_source_train.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 1E-2: Target class distribution (individual)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
cnt = [tgt_train_idx.labels.count(i) for i in range(NUM_CLASSES)]
bars = ax.bar(range(NUM_CLASSES), cnt, color='#FF5722', alpha=.85)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([c[:20] for c in CLASSES], rotation=15, fontsize=9)
ax.set_title('Target Train — Class Distribution', fontweight='bold')
ax.set_ylabel('Number of Images')
for i, v in enumerate(cnt):
    ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_dist_target_train.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 1E-3: Source class share (pie)
# ============================================================
fig, ax = plt.subplots(figsize=(6, 6))
src_cnt = [src_index.labels.count(i) for i in range(NUM_CLASSES)]
ax.pie(src_cnt, labels=[c[:18] for c in CLASSES], autopct='%1.1f%%',
       colors=['#E53935', '#1E88E5', '#43A047'], startangle=90)
ax.set_title('Source (PlantVillage) — Class Share', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_pie_source.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 1E-4: Target class share (pie)
# ============================================================
fig, ax = plt.subplots(figsize=(6, 6))
tgt_cnt = [tgt_index.labels.count(i) for i in range(NUM_CLASSES)]
ax.pie(tgt_cnt, labels=[c[:18] for c in CLASSES], autopct='%1.1f%%',
       colors=['#E53935', '#1E88E5', '#43A047'], startangle=90)
ax.set_title('Target (PlantDoc) — Class Share', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_pie_target.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 1E-5: Image size (height, width) distribution
# ============================================================
import random as _rnd
_rng = _rnd.Random(SEED)

# Source sizes
src_sizes = []
src_sample = _rng.sample(range(len(src_index.img_paths)), min(200, len(src_index.img_paths)))
for si in src_sample:
    try:
        w, h = Image.open(src_index.img_paths[si]).size
        src_sizes.append((w, h))
    except: pass
sw, sh = zip(*src_sizes) if src_sizes else ([], [])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sw, sh, c='#2196F3', alpha=0.6, s=30, label='Source')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.set_title('Source Image Dimensions (sample)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/img_size_source.png", dpi=150, bbox_inches='tight')
plt.show()

# Target sizes
tgt_sizes = []
tgt_sample = _rng.sample(range(len(tgt_index.img_paths)), min(200, len(tgt_index.img_paths)))
for si in tgt_sample:
    try:
        w, h = Image.open(tgt_index.img_paths[si]).size
        tgt_sizes.append((w, h))
    except: pass
tw, th = zip(*tgt_sizes) if tgt_sizes else ([], [])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(tw, th, c='#FF5722', alpha=0.6, s=30, label='Target')
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.set_title('Target Image Dimensions (sample)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/img_size_target.png", dpi=150, bbox_inches='tight')
plt.show()

---
# Part 2: Architecture

Same three-stage idea as v6 — CNN → Squeeze-Excite → Micro-ViT with linear attention — with
three changes:

- **Wider and higher-resolution (FIX #8).** `(32,48,64,96,96)` at 128px instead of
  `(16,24,32,48,48)` at 96px -- 167K params, ~163 KB INT8. v6 shipped 42 KB against a stated 500 KB budget; the two
  classes it confused (Septoria's small scattered spots vs Early Blight's concentric rings) are
  separated by exactly the fine texture that 96px destroys. The extra stride-2 stage keeps the
  token grid at 8×8 = 64, so linear attention stays cheap.
- **Learned positional embedding.** v6's Micro-ViT had none, so its attention was
  permutation-invariant over patches — lesion *distribution* across the leaf was invisible to it.
- **`kd_proj` lives outside the model.** The feature-distillation projection is a separate
  module, so the deployed graph is untouched and nothing needs stripping before export.

`extract_features()` is the DA hook: CORAL/MMD/IM/MCC all operate on its output. (This run has `USE_CORAL=USE_MMD=USE_MCC=False`, `USE_IM=True` -- see Part 0 -- so only IM actually touches it here.)

In [ ]:
# ============================================================
# PART 2: MicroBrainNet-DA v7
# ============================================================

class InvertedResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand_ratio=2):
        super().__init__()
        self.use_residual = (stride == 1 and in_ch == out_ch)
        hid = in_ch * expand_ratio
        layers = []
        if expand_ratio != 1:
            layers += [nn.Conv2d(in_ch, hid, 1, bias=False), nn.BatchNorm2d(hid),
                       nn.ReLU6(inplace=True)]
        layers += [nn.Conv2d(hid, hid, 3, stride, 1, groups=hid, bias=False),
                   nn.BatchNorm2d(hid), nn.ReLU6(inplace=True),
                   nn.Conv2d(hid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch)]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.conv(x) if self.use_residual else self.conv(x)


class SqueezeExciteBlock(nn.Module):
    def __init__(self, ch, reduction=4):
        super().__init__()
        mid = max(ch // reduction, 8)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(nn.Linear(ch, mid), nn.ReLU(inplace=True),
                                    nn.Linear(mid, ch), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.excite(self.squeeze(x).view(b, c)).view(b, c, 1, 1)


class MicroViTBlock(nn.Module):
    """Linear attention, O(n). ReLU6 in the FFN (not GELU) so every op has a TFLite Micro kernel."""

    def __init__(self, dim, num_tokens, ffn_ratio=2):
        super().__init__()
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim); self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        # learned positional embedding -- v6 had none, so patch layout was invisible
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        h = dim * ffn_ratio
        self.ffn = nn.Sequential(nn.Linear(dim, h), nn.ReLU6(inplace=True), nn.Dropout(0.1),
                                 nn.Linear(h, dim), nn.Dropout(0.1))

    def linear_attention(self, q, k, v):
        q = F.elu(q) + 1
        k = F.elu(k) + 1
        kv = torch.bmm(k.transpose(1, 2), v)
        num = torch.bmm(q, kv)
        den = torch.bmm(q, k.sum(1, keepdim=True).transpose(1, 2)).clamp(min=1e-4)
        return num / den

    def forward(self, x):
        x = x + self.pos
        xn = self.norm1(x)
        x = x + self.out_proj(self.linear_attention(self.q_proj(xn), self.k_proj(xn),
                                                    self.v_proj(xn)))
        return x + self.ffn(self.norm2(x))


class MicroBrainNetDA(nn.Module):
    """128x128x3 RGB -> 3 logits. Downsampling: stem s2, b1 s2, b2 s2, b3 s2, b4 s1
    => 8x8 = 64 tokens for the ViT stage."""

    def __init__(self, num_classes=NUM_CLASSES, channels=STUDENT_CHANNELS, img_size=IMG_SIZE,
                 use_se=True, use_vit=True):
        super().__init__()
        c0, c1, c2, c3, c4 = channels
        self.use_se, self.use_vit = use_se, use_vit
        self.stem = nn.Sequential(nn.Conv2d(3, c0, 3, 2, 1, bias=False),
                                  nn.BatchNorm2d(c0), nn.ReLU6(inplace=True))
        self.block1 = InvertedResidualBlock(c0, c1, 2, 2)
        self.block2 = InvertedResidualBlock(c1, c2, 2, 2)
        self.block3 = InvertedResidualBlock(c2, c3, 2, 2)
        self.block4 = InvertedResidualBlock(c3, c4, 1, 2)
        self.se = SqueezeExciteBlock(c4) if use_se else nn.Identity()
        self.grid = img_size // 16
        self.vit = MicroViTBlock(c4, self.grid * self.grid) if use_vit else nn.Identity()
        self.classifier = nn.Sequential(nn.LayerNorm(c4), nn.Dropout(0.2),
                                        nn.Linear(c4, num_classes))
        self.feature_dim = c4
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def extract_features(self, x):
        x = self.stem(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        x = self.se(x)
        if self.use_vit:
            x = x.flatten(2).transpose(1, 2)
            x = self.vit(x)
            return x.mean(1)
        return F.adaptive_avg_pool2d(x, 1).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


class TeacherNet(nn.Module):
    """ImageNet-pretrained backbone behind the same extract_features()/classifier interface,
    so it drops straight into train_da_epoch(). Training-time only -- never quantized."""

    def __init__(self, num_classes=NUM_CLASSES, backbone_name=TEACHER_BACKBONE):
        super().__init__()
        if backbone_name == 'mobilenet_v3_large':
            bb = tvm.mobilenet_v3_large(weights=tvm.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        elif backbone_name == 'mobilenet_v3_small':
            bb = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        elif backbone_name == 'efficientnet_b0':
            bb = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            raise ValueError(backbone_name)
        self.features = bb.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feature_dim = bb.classifier[0].in_features
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.feature_dim, num_classes))

    def extract_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


model = MicroBrainNetDA().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Student params : {total_params:,}")
print(f"  FP32 : {total_params*4/1024:.1f} KB")
print(f"  INT8 : {total_params/1024:.1f} KB   (budget 500 KB -> "
      f"{total_params/1024/500*100:.0f}% used; v6 used 8%)")
print(f"  ViT token grid: {model.grid}x{model.grid} = {model.grid**2} tokens")
with torch.no_grad():
    print("forward:", tuple(model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)).shape))

Student params : 179,803
  FP32 : 702.4 KB
  INT8 : 175.6 KB   (budget 500 KB -> 35% used; v6 used 8%)
  ViT token grid: 14x14 = 196 tokens
forward: (2, 3)


In [ ]:
# ============================================================
# PART 2-R: ResNet-50 STUDENT (replaces MicroBrainNetDA as the student)
# ============================================================
# Same interface as MicroBrainNetDA / TeacherNet: extract_features(), classifier, feature_dim,
# so train_da_epoch(), KD, EMA, AdaBN, evaluate() all work unchanged.
# Input is the pipeline's student view (Normalize(0.5, 0.5) -> [-1, 1]); the wrapper converts it to
# ImageNet normalisation INSIDE the model so (a) pretrained weights see what they expect and
# (b) the exported TFLite model can take raw [0,1] pixels exactly like the original export did.

class ResNet50Student(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=STUDENT_PRETRAINED):
        super().__init__()
        bb = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool,
                                      bb.layer1, bb.layer2, bb.layer3, bb.layer4)
        self.feature_dim = bb.fc.in_features            # 2048
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.feature_dim, num_classes))
        self.register_buffer('_mean', torch.tensor(list(IMAGENET_MEAN)).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('_std',  torch.tensor(list(IMAGENET_STD)).view(1, 3, 1, 1),  persistent=False)
        for m in self.modules():                        # Grad-CAM backward hooks break on in-place ReLU
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def extract_features(self, x):
        x = (x * 0.5 + 0.5 - self._mean) / self._std    # [-1,1] -> ImageNet-normalised
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))


def build_student():
    return ResNet50Student() if STUDENT_ARCH == 'resnet50' else MicroBrainNetDA()


model = build_student().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Student = {STUDENT_ARCH} | pretrained={STUDENT_PRETRAINED} | input {IMG_SIZE}px")
print(f"Student params : {total_params:,}  ({total_params/1e6:.2f} M)")
print(f"  FP32 : {total_params*4/1024:.1f} KB   INT8 ~ {total_params/1024:.1f} KB   (500 KB budget -> NOT met, expected for ResNet-50)")
with torch.no_grad():
    model.eval()
    print("forward:", tuple(model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)).shape))
    model.train()


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]


Student = resnet50 | pretrained=True | input 224px
Student params : 23,514,179  (23.51 M)
  FP32 : 91852.3 KB   INT8 ~ 22963.1 KB   (500 KB budget -> NOT met, expected for ResNet-50)
forward: (2, 3)


In [ ]:
# ============================================================
# VIZ 2-1: Parameter count per module (individual bar)
# ============================================================
param_counts = {}
for name, p in model.named_parameters():
    module = name.split('.')[0]
    param_counts[module] = param_counts.get(module, 0) + p.numel()

fig, ax = plt.subplots(figsize=(10, 5))
modules = list(param_counts.keys())
counts = list(param_counts.values())
bars = ax.barh(modules, counts, color='#7B1FA2', alpha=0.85)
ax.set_xlabel('Number of Parameters')
ax.set_title('Parameter Distribution Across Model Modules', fontweight='bold')
for b in bars:
    ax.text(b.get_width(), b.get_y() + b.get_height()/2,
            f' {int(b.get_width()):,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/param_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 2-2: Model size budget visualization
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
sizes = {'INT8 Model': total_params / 1024, 'Budget (500 KB)': 500}
colors = ['#4CAF50', '#B0BEC5']
bars = ax.barh(list(sizes.keys()), list(sizes.values()), color=colors, height=0.5)
ax.set_xlabel('Size (KB)')
ax.set_title('Model Size vs Budget', fontweight='bold')
for b in bars:
    ax.text(b.get_width() + 5, b.get_y() + b.get_height()/2,
            f'{b.get_width():.1f} KB', va='center', fontweight='bold')
ax.set_xlim(0, max(600, total_params / 1024 * 1.15))
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/model_size_budget.png", dpi=150, bbox_inches='tight')
plt.show()

---
# Part 3: Losses

### `im_loss` replaces bare entropy minimization (FIX #3)

This is the single most important line in the notebook. Entropy minimization alone has one
trivially optimal solution: send every target sample to the same class. That is precisely v6's
failure — Potato vanished and source val froze at `0.4715 + 0.2622 = 0.7337`. Information
Maximization keeps the confidence term but subtracts the entropy of the *batch-mean* prediction,
so a collapsed solution is now maximally penalised rather than rewarded.

### `mcc_loss` added

CORAL and MMD are class-agnostic: they are perfectly satisfied by any feature map that matches
the two domains' statistics, including one that has thrown away class structure — which is what
`coral: 0.0121 -> 0.0001` with flat accuracy meant in v6. Minimum Class Confusion penalises
off-diagonal mass in the target's own class-confusion matrix, so it can only be reduced by
making target predictions *mutually distinct*.

### CORAL/MMD kept, but demoted

Still useful as a weak prior; weights cut from 5.0/1.0 to 0.5/0.5, and Part 5 flags the run if
CORAL collapses while target accuracy stays flat.

### `balanced_pseudo` replaces global-threshold pseudo-labeling (FIX #1)

v6 took every prediction above 0.98 confidence, which at 50-60% correctness meant ~2450 mostly
wrong labels per epoch over 364 images. Three changes: labels come from an **EMA** copy of the
model (a target moving slower than the student cannot chase its own errors as tightly); selection
is **class-balanced top-k** so no single class can dominate; and the caller enforces a hard
**auto-off** when measured pseudo-label accuracy drops below 0.85.

### This run: IM-only

`CORAL`, `MMD` and `MCC` below are fully implemented and are what v7 normally combines with IM, but for this run `USE_CORAL = USE_MMD = USE_MCC = False` (Part 0) so their weights are forced to 0 in `train_da_epoch` and none of them contribute to the loss. Information Maximization (`im_loss`) is the only feature/label-alignment term doing any work -- everything downstream (student, teacher, ablation) reflects that.

In [ ]:
# ============================================================
# PART 3: LOSS FUNCTIONS
# ============================================================

def standardize_feats(x, eps=1e-5):
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + eps)


def compute_covariance(x):
    n = x.size(0)
    ones = torch.ones(1, n, device=x.device)
    mean = ones @ x / n
    return (x.t() @ x - mean.t() @ mean) / max(n - 1, 1)


def coral_loss(fs, ft):
    s, t = standardize_feats(fs), standardize_feats(ft)
    d = s.size(1)
    return (compute_covariance(s) - compute_covariance(t)).pow(2).sum() / (4.0 * d * d)


def gaussian_kernel_matrix(s, t, kernel_mul=2.0, kernel_num=5):
    total = torch.cat([s, t], 0)
    n = total.size(0)
    l2 = ((total.unsqueeze(0) - total.unsqueeze(1)) ** 2).sum(2)
    bw = torch.sum(l2.detach()) / max(n * n - n, 1)
    bw = bw / (kernel_mul ** (kernel_num // 2)) + 1e-8
    return sum(torch.exp(-l2 / (bw * kernel_mul ** i)) for i in range(kernel_num))


def mmd_loss(fs, ft):
    s, t = standardize_feats(fs), standardize_feats(ft)
    ns = s.size(0)
    k = gaussian_kernel_matrix(s, t)
    return k[:ns, :ns].mean() + k[ns:, ns:].mean() - k[:ns, ns:].mean() - k[ns:, :ns].mean()


def im_loss(logits):
    """Information Maximization = confident per sample MINUS diverse across the batch.

    FIX #3. Bare entropy minimization (v6) has a degenerate optimum: predict one class for
    everything. The batch-marginal term makes that the WORST solution instead of the best.
    Returns ent_individual - ent_batch; both terms are minimized together."""
    p = F.softmax(logits, 1)
    ent_ind = -(p * torch.log(p + 1e-8)).sum(1).mean()
    pm = p.mean(0)
    ent_batch = -(pm * torch.log(pm + 1e-8)).sum()
    return ent_ind - ent_batch


def mcc_loss(logits, T=2.5):
    """Minimum Class Confusion (Jin et al.). Class-aware, so unlike CORAL/MMD it cannot be
    satisfied by a feature collapse -- off-diagonal mass only falls if target predictions
    become mutually distinct."""
    p = F.softmax(logits / T, 1)
    w = (1 + (-(p * torch.log(p + 1e-8)).sum(1))).detach()
    w = (w / w.sum()) * p.size(0)
    C = (p * w.unsqueeze(1)).t() @ p
    C = C / (C.sum(1, keepdim=True) + 1e-8)
    return (C.sum() - C.trace()) / logits.size(1)


def kd_loss_fn(s_logits, t_logits, T=4.0):
    return F.kl_div(F.log_softmax(s_logits / T, 1), F.softmax(t_logits / T, 1),
                    reduction='batchmean') * (T * T)


def feat_kd_loss(s_proj, t_feat):
    """Cosine feature distillation. With a ~70x capacity gap, logit-only KD transfers little;
    matching normalized feature direction gives the student a much denser signal."""
    return F.mse_loss(F.normalize(s_proj, dim=1), F.normalize(t_feat.detach(), dim=1))


@torch.no_grad()
def balanced_pseudo(logits, k_per_class=PSEUDO_TOPK_PER_CLASS, min_conf=PSEUDO_CONF,
                    n_classes=NUM_CLASSES):
    """Class-balanced top-k selection (FIX #1). A global threshold lets the dominant class
    supply nearly every pseudo-label, which is how v6 reinforced its own Potato->Tomato error."""
    probs = F.softmax(logits, 1)
    conf, lab = probs.max(1)
    keep = torch.zeros_like(conf, dtype=torch.bool)
    for c in range(n_classes):
        idx = ((lab == c) & (conf >= min_conf)).nonzero(as_tuple=True)[0]
        if idx.numel():
            top = conf[idx].topk(min(k_per_class, idx.numel())).indices
            keep[idx[top]] = True
    return keep, lab


def ramp(epoch, max_epoch, warm=0):
    """Sigmoid ramp for the alignment weights. KD is deliberately NOT ramped (FIX #4):
    ramping it up meant it peaked exactly when it should have been fading out."""
    if epoch <= warm:
        return 0.0
    p = (epoch - warm) / max(max_epoch - warm, 1)
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


print("Losses ready: coral, mmd, im (IM replaces entropy-min), mcc, kd, feat_kd, balanced_pseudo")

Losses ready: coral, mmd, im (IM replaces entropy-min), mcc, kd, feat_kd, balanced_pseudo


---
# Part 4: Training utilities

### The BatchNorm fix (FIX #5)

v6 called `extract_features(source)` and `extract_features(target)` as two separate forward
passes in `train()` mode. Each pass updated the BN running statistics, so despite target being
12% of the data, the deployed running mean/var ended up a roughly 50/50 blend of two very
different distributions — and every `eval()` measurement, on both domains, was made through
those wrong statistics. Here the two batches are **concatenated into a single forward** and split
afterwards, so BN sees one coherent mixed-domain distribution and updates once per step.

`adapt_bn()` (AdaBN) then goes further: before final evaluation and before INT8 calibration it
resets the running statistics and recomputes them from target data only. In published benchmarks
of this exact shift, adaptive batch normalization was one of the few mitigations that reliably
helped, while adversarial adaptation actively hurt.

### The target over-sampling fix (FIX #2)

`n_da_steps = min(len(src_loader), DA_TARGET_PASSES * len(tgt_loader))`. Beyond that many steps
the epoch continues on source CE (plus source-side KD) alone. In v6 all alignment terms ran on
every one of the 41 source steps while the target loader held only 5 batches, so 364 images were
replayed 8.2 times per epoch — enough to satisfy CORAL by memorising them.

### Early stopping

Uses a 5-epoch rolling mean of target-val accuracy. On 79 images a single epoch's raw number
moves 1.27 points per image, so unsmoothed selection just latches onto a lucky epoch.

In [ ]:
# ============================================================
# PART 4: TRAINING UTILITIES
# ============================================================

class EMA:
    """Exponential moving average of the student, used only to generate pseudo-labels."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best, self.early_stop, self.best_model = 0, None, False, None

    def __call__(self, score, model):
        if self.best is None or score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            self.best_model = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


@torch.no_grad()
def evaluate(model, loader, criterion, device, which='s'):
    """which='s' -> 128px/[-1,1] student view; which='t' -> 224px/ImageNet teacher view."""
    model.eval()
    tot, preds, labs, probs = 0.0, [], [], []
    for xs, xt, xw_s, xw_t, y, d in loader:
        x = (xs if which == 's' else xt).to(device)
        y = y.to(device)
        logits = model(x)
        tot += criterion(logits, y).item() * x.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
        probs.extend(F.softmax(logits, 1).cpu().numpy())
    n = len(labs)
    return (tot / n, accuracy_score(labs, preds), f1_score(labs, preds, average='weighted'),
            np.array(preds), np.array(labs), np.array(probs))


@torch.no_grad()
def adapt_bn(model, loader, device, which='s', n_pass=3):
    """AdaBN (FIX #5): recompute BN running statistics from TARGET data only.
    momentum=None makes each BN accumulate a cumulative average over the passes.
    Must run before INT8 calibration, otherwise quantization bakes in the wrong statistics."""
    has_bn = False
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.reset_running_stats(); m.momentum = None; has_bn = True
    if not has_bn:
        return model
    model.train()
    for _ in range(n_pass):
        for xs, xt, xw_s, xw_t, y, d in loader:
            model((xs if which == 's' else xt).to(device))
    model.eval()
    return model


def train_da_epoch(model, src_loader, tgt_loader, criterion, optimizer, device,
                   epoch, max_epoch, cfg, which='s',
                   teacher=None, kd_proj=None, ema=None,
                   pseudo_enabled=False,
                   use_coral=True, use_mmd=True, use_im=True, use_mcc=True, use_kd=True):
    """One epoch of domain-adaptive (+ optionally distilled) training.

    Key structural differences from v6:
      * source and target go through ONE concatenated forward -> clean BN statistics (FIX #5)
      * alignment losses run on at most cfg['da_passes'] * len(tgt_loader) steps (FIX #2)
      * KD uses the paired native-resolution teacher view from DualResDataset (FIX #4)
      * pseudo-labels come from the EMA model on the WEAK view, applied to the STRONG view
    """
    model.train()
    lam = ramp(epoch, max_epoch)
    w = dict(coral=cfg['coral'] * lam if use_coral else 0.0,
             mmd=cfg['mmd'] * lam if use_mmd else 0.0,
             im=cfg['im'] * lam if use_im else 0.0,
             mcc=cfg['mcc'] * lam if use_mcc else 0.0,
             pseudo=cfg['pseudo'] * lam if pseudo_enabled else 0.0,
             kd=cfg['kd'] if (use_kd and teacher is not None) else 0.0,
             featkd=cfg['featkd'] if (use_kd and teacher is not None and kd_proj is not None) else 0.0)

    n_da = min(len(src_loader), max(1, cfg['da_passes'] * len(tgt_loader)))
    tgt_iter = iter(tgt_loader)

    acc_loss = {k: 0.0 for k in ['cls', 'coral', 'mmd', 'im', 'mcc', 'pseudo', 'kd', 'featkd']}
    n_seen, n_da_seen = 0, 0
    preds, labs = [], []
    ps_used, ps_correct = 0, 0

    for step, (xs_s, xs_t, _w1, _w2, ys, _d) in enumerate(src_loader):
        do_da = step < n_da
        x_src = (xs_s if which == 's' else xs_t).to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)
        bs = x_src.size(0)

        if do_da:
            try:
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt_s, xt_t, xt_ws, xt_wt, yt_true, _ = next(tgt_iter)
            x_tgt = (xt_s if which == 's' else xt_t).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ---- single concatenated forward: clean BN statistics (FIX #5) ----
        if do_da:
            feats = model.extract_features(torch.cat([x_src, x_tgt], 0))
            f_s, f_t = feats[:bs], feats[bs:]
            logits_s, logits_t = model.classifier(f_s), model.classifier(f_t)
        else:
            f_s = model.extract_features(x_src)
            f_t, logits_t = None, None
            logits_s = model.classifier(f_s)

        L_cls = criterion(logits_s, ys)
        loss = L_cls
        parts = {k: 0.0 for k in acc_loss}
        parts['cls'] = L_cls.item()

        if do_da:
            if w['coral'] > 0:
                l = coral_loss(f_s, f_t);  loss = loss + w['coral'] * l;  parts['coral'] = l.item()
            if w['mmd'] > 0:
                l = mmd_loss(f_s, f_t);    loss = loss + w['mmd'] * l;    parts['mmd'] = l.item()
            if w['im'] > 0:
                l = im_loss(logits_t);     loss = loss + w['im'] * l;     parts['im'] = l.item()
            if w['mcc'] > 0:
                l = mcc_loss(logits_t);    loss = loss + w['mcc'] * l;    parts['mcc'] = l.item()

            if w['pseudo'] > 0 and ema is not None:
                with torch.no_grad():
                    # weak view at the resolution THIS model consumes -> pseudo-labels
                    xw = (xt_ws if which == 's' else xt_wt).to(device)
                    weak_logits = ema.model(xw)
                keep, plab = balanced_pseudo(weak_logits, cfg['pseudo_topk'], cfg['pseudo_conf'])
                if keep.sum() > 0:
                    l = F.cross_entropy(logits_t[keep], plab[keep])  # strong view -> trained on
                    loss = loss + w['pseudo'] * l
                    parts['pseudo'] = l.item()
                    ps_used += int(keep.sum())
                    ps_correct += int((plab[keep].cpu() == yt_true[keep.cpu()]).sum())

        # ---- distillation on the paired native-resolution teacher view (FIX #4) ----
        if w['kd'] > 0:
            with torch.no_grad():
                tf_s = teacher.extract_features(xs_t.to(device, non_blocking=True))
                tl_s = teacher.classifier(tf_s)
                if do_da:
                    tf_t = teacher.extract_features(xt_t.to(device, non_blocking=True))
                    tl_t = teacher.classifier(tf_t)
            l = kd_loss_fn(logits_s, tl_s, cfg['T'])
            if do_da:
                l = 0.5 * (l + kd_loss_fn(logits_t, tl_t, cfg['T']))
            loss = loss + w['kd'] * l
            parts['kd'] = l.item()

            if w['featkd'] > 0:
                lf = feat_kd_loss(kd_proj(f_s), tf_s)
                if do_da:
                    lf = 0.5 * (lf + feat_kd_loss(kd_proj(f_t), tf_t))
                loss = loss + w['featkd'] * lf
                parts['featkd'] = lf.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if ema is not None:
            ema.update(model)

        for k in acc_loss:
            acc_loss[k] += parts[k] * bs
        n_seen += bs
        n_da_seen += bs if do_da else 0
        preds.extend(logits_s.argmax(1).detach().cpu().numpy())
        labs.extend(ys.cpu().numpy())

    out = {f'{k}_loss': v / max(n_seen, 1) for k, v in acc_loss.items()}
    out.update(train_acc=accuracy_score(labs, preds),
               train_f1=f1_score(labs, preds, average='weighted'),
               lam=lam, pseudo_used=ps_used,
               pseudo_acc=(ps_correct / ps_used) if ps_used else float('nan'),
               weights=w)
    return out


def make_scheduler(opt, epochs, warmup):
    def fn(e):
        return (e + 1) / max(warmup, 1) if e < warmup else \
               0.5 * (1 + math.cos(math.pi * (e - warmup) / max(epochs - warmup, 1)))
    return optim.lr_scheduler.LambdaLR(opt, fn)


cls_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler")

Utilities ready: EMA, EarlyStopping, evaluate, adapt_bn, train_da_epoch, make_scheduler


---
# Part 0B: Sanity gate — can the model fit SOURCE at all?

**Run this before anything else.** It trains the student and (optionally) a wide variant on
source only: no CORAL, no MMD, no IM, no MCC, no pseudo-labels, no distillation. It answers the
one question that determines whether any downstream tuning is meaningful.

| Result | Meaning | Next step |
|---|---|---|
| Src-Val **≥ 0.97** | Architecture and optimizer are fine. v6's 0.730 was caused by the training recipe, exactly as diagnosed. | Continue to Part 5. |
| Src-Val ≈ **0.73** (or any `1 - class_share` value printed in Part 1E) | A class is being erased even without DA. The problem is architectural or in the data. | Stop. Inspect the confusion matrix printed below before training anything else. |

The cell also prints train accuracy alongside val accuracy. v6 never logged the TA's train
accuracy, which is why it was impossible to tell "cannot fit the data" apart from "fits fine but
BN statistics are wrong at eval time" — the two require completely different fixes.

In [ ]:
# ============================================================
# PART 0B: SANITY GATE -- SOURCE-ONLY, NO DA, NO KD
# ============================================================
if RUN_SANITY_GATE:
    set_seed(SEED)
    print("=" * 72)
    print("  SANITY GATE: source-only training. Target >= 0.97 source-val accuracy.")
    print("=" * 72)

    sg_model = build_student().to(DEVICE)
    sg_opt = optim.AdamW(sg_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sg_sched = make_scheduler(sg_opt, 30, 3)
    sg_crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    for ep in range(1, 31):
        sg_model.train()
        p_, l_ = [], []
        for xs, xt, xw_s, xw_t, y, d in src_train_loader:
            xs, y = xs.to(DEVICE), y.to(DEVICE)
            sg_opt.zero_grad(set_to_none=True)
            out = sg_model(xs)
            loss = sg_crit(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sg_model.parameters(), 1.0)
            sg_opt.step()
            p_.extend(out.argmax(1).detach().cpu().numpy()); l_.extend(y.cpu().numpy())
        sg_sched.step()
        if ep % 5 == 0 or ep == 1:
            tr = accuracy_score(l_, p_)
            _, sv, _, _, _, _ = evaluate(sg_model, src_val_loader, sg_crit, DEVICE)
            _, tv, _, _, _, _ = evaluate(sg_model, tgt_val_loader, sg_crit, DEVICE)
            print(f"  Ep {ep:2d}/30 | TRAIN acc={tr:.4f} | SRC-VAL acc={sv:.4f} | TGT-VAL acc={tv:.4f}")

    _, sv, sf, sp, sl, _ = evaluate(sg_model, src_val_loader, sg_crit, DEVICE)
    _, tv, tf, tp, tl, _ = evaluate(sg_model, tgt_val_loader, sg_crit, DEVICE)

    print(f"\n  FINAL  SRC-VAL acc = {sv:.4f}   f1 = {sf:.4f}")
    print(f"         TGT-VAL acc = {tv:.4f}   f1 = {tf:.4f}  (source-only baseline, no DA)")
    print("\n  Source-val confusion matrix (rows = actual):")
    cm = confusion_matrix(sl, sp)
    print("  " + " " * 26 + "".join(f"{c[:13]:>15s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"  {c:<26s}" + "".join(f"{cm[i][j]:>15d}" for j in range(NUM_CLASSES)))

    print("\n  " + "-" * 60)
    if sv >= 0.97:
        print("  VERDICT: PASS. Architecture and optimizer are healthy.")
        print("           v6's 0.730 plateau came from the recipe (pseudo-label poisoning,")
        print("           entropy collapse, KD on blurry inputs) -- all addressed below.")
    elif sv >= 0.90:
        print("  VERDICT: MARGINAL. Not collapse, but under-fitting. Try LR 2e-3 / 50 epochs")
        print("           before moving on, and check the augmentation preview in Part 1E.")
    else:
        print("  VERDICT: FAIL. A class is still being erased WITHOUT any DA loss.")
        print("           Do not proceed. Check the confusion matrix above: if one row is")
        print("           empty, inspect the class folders and the augmentation preview.")
    print("  " + "-" * 60)

    sanity_src_val, sanity_tgt_val = sv, tv
    del sg_model, sg_opt
    torch.cuda.empty_cache()
else:
    print("RUN_SANITY_GATE=False -- skipped. Run it at least once on a fresh environment.")
    sanity_src_val = sanity_tgt_val = None

  SANITY GATE: source-only training. Target >= 0.97 source-val accuracy.
  Ep  1/30 | TRAIN acc=0.9313 | SRC-VAL acc=1.0000 | TGT-VAL acc=0.8354
  Ep  5/30 | TRAIN acc=0.9765 | SRC-VAL acc=0.9965 | TGT-VAL acc=0.7215
  Ep 10/30 | TRAIN acc=0.9850 | SRC-VAL acc=0.9965 | TGT-VAL acc=0.6709
  Ep 15/30 | TRAIN acc=0.9896 | SRC-VAL acc=1.0000 | TGT-VAL acc=0.7595
  Ep 20/30 | TRAIN acc=0.9934 | SRC-VAL acc=1.0000 | TGT-VAL acc=0.7089
  Ep 25/30 | TRAIN acc=0.9973 | SRC-VAL acc=1.0000 | TGT-VAL acc=0.7215
  Ep 30/30 | TRAIN acc=0.9981 | SRC-VAL acc=1.0000 | TGT-VAL acc=0.7089

  FINAL  SRC-VAL acc = 1.0000   f1 = 1.0000
         TGT-VAL acc = 0.7089   f1 = 0.6866  (source-only baseline, no DA)

  Source-val confusion matrix (rows = actual):
                              Tomato_Septor  Potato_Early_  Corn_Leaf_Bli
  Tomato_Septoria_Leaf_Spot             266              0              0
  Potato_Early_Blight                     0            150              0
  Corn_Leaf_Blight               

In [ ]:
# ============================================================
# VIZ 0B-1: Sanity gate confusion matrix (individual)
# ============================================================
if RUN_SANITY_GATE and 'sg_model' not in dir():
    print("Sanity gate model was deleted. Showing final numbers only.")
    print(f"  Source Val Acc: {sanity_src_val:.4f}")
    print(f"  Target Val Acc: {sanity_tgt_val:.4f}")
elif RUN_SANITY_GATE:
    print(f"Sanity Gate Results:")
    print(f"  Source Val Acc: {sanity_src_val:.4f}")
    print(f"  Target Val Acc: {sanity_tgt_val:.4f}")

Sanity gate model was deleted. Showing final numbers only.
  Source Val Acc: 1.0000
  Target Val Acc: 0.7089


---
# Part 5: Teacher

The teacher is the accuracy ceiling for everything downstream, so it is fixed first. v6's teacher
peaked at 0.7975 raw / 0.7753 smoothed and early-stopped at epoch 24. Changes:

- **Pseudo-labeling off by default (FIX #1).** The v6 log is unambiguous: target val ran
  `0.722 (pseudo off) -> 0.734 -> 0.671` once self-training engaged at ~50-60% label accuracy.
  Establish the clean number first. Only then consider `TEACHER_USE_PSEUDO=True`, which now goes
  through the EMA + class-balanced + auto-off path.
- **Discriminative learning rates (FIX #10).** v6 froze `TEACHER_FREEZE_BLOCKS=4`, which froze
  8,808 of 2,974,835 parameters — 0.3%, cosmetic. Real control comes from a low backbone LR
  (3e-4) with a high head LR (3e-3) plus warmup, so the randomly initialised head converges
  without dragging the pretrained features with it.
- **Patience 30, 80 epochs.** With warmup + cosine there is no reason to stop at 24.
- **Same strong augmentation** as the student, at native 224px.

Watch `TRAIN` alongside `SRC-VAL`: source saturating near 0.99 in the first epochs is expected and
is precisely why the augmentation and the target-side terms have to carry the run.

In [ ]:
# ============================================================
# PART 5: TEACHER FINE-TUNING
# ============================================================
if USE_TEACHER_KD:
    set_seed(SEED)
    teacher = TeacherNet().to(DEVICE)
    t_params = sum(p.numel() for p in teacher.parameters())
    print(f"Teacher: {TEACHER_BACKBONE}, {t_params:,} params (training-time only, never exported)")

    # FIX #10: discriminative LR instead of a cosmetic block freeze
    head_ids = {id(p) for p in teacher.classifier.parameters()}
    t_opt = optim.AdamW([
        {'params': [p for p in teacher.parameters() if id(p) not in head_ids],
         'lr': TEACHER_LR_BACKBONE},
        {'params': list(teacher.classifier.parameters()), 'lr': TEACHER_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    t_sched = make_scheduler(t_opt, TEACHER_EPOCHS, TEACHER_WARMUP)
    t_stop = EarlyStopping(patience=TEACHER_PATIENCE, min_delta=1e-3)
    t_ema = EMA(teacher) if TEACHER_USE_PSEUDO else None

    t_hist = {'src_val': [], 'tgt_val': [], 'train': [], 'coral': [], 'mcc': [], 'im': [], 'train_loss': [], 'val_loss': []}
    t_raw, t_best_raw, t_best_sm, t_best_state = [], 0.0, 0.0, copy.deepcopy(teacher.state_dict())
    t_pseudo_on, t_last_ps_acc = False, float('nan')
    t_plateau = 0
    t0 = time.time()

    print("=" * 96)
    print(f"  Teacher | {TEACHER_EPOCHS} epochs | 224px | DA=IM-only(CORAL={USE_CORAL} MMD={USE_MMD} MCC={USE_MCC}) | pseudo-labeling="
          f"{'ON' if TEACHER_USE_PSEUDO else 'OFF (v6 poisoned itself here)'}")
    print("=" * 96)

    for ep in range(1, TEACHER_EPOCHS + 1):
        st = train_da_epoch(teacher, src_train_loader, tgt_train_loader, cls_criterion,
                            t_opt, DEVICE, ep, TEACHER_EPOCHS, CFG, which='t',
                            teacher=None, kd_proj=None, ema=t_ema,
                            pseudo_enabled=t_pseudo_on, use_kd=False,
                            use_coral=USE_CORAL, use_mmd=USE_MMD, use_im=USE_IM, use_mcc=USE_MCC)
        t_sched.step()

        t_val_loss_src, sv, _, _, _, _ = evaluate(teacher, src_val_loader, cls_criterion, DEVICE, 't')
        t_val_loss_tgt, tv, tf1, _, _, _ = evaluate(teacher, tgt_val_loader, cls_criterion, DEVICE, 't')

        t_raw.append(tv)
        sm = float(np.mean(t_raw[-SMOOTH_WINDOW:]))
        t_hist['src_val'].append(sv); t_hist['tgt_val'].append(tv)
        t_hist['train'].append(st['train_acc']); t_hist['train_loss'].append(st['cls_loss']); t_hist['val_loss'].append(t_val_loss_src); t_hist['coral'].append(st['coral_loss'])
        t_hist['mcc'].append(st['mcc_loss']); t_hist['im'].append(st['im_loss'])

        t_best_raw = max(t_best_raw, tv)
        if sm > t_best_sm:
            t_best_sm = sm
            t_best_state = copy.deepcopy(teacher.state_dict())
        t_stop(sm, teacher)

        # ---- pseudo-label control (FIX #1): plateau gate + hard auto-off ----
        if TEACHER_USE_PSEUDO:
            if len(t_raw) > SMOOTH_WINDOW and sm <= t_best_sm:
                t_plateau += 1
            else:
                t_plateau = 0
            if (not t_pseudo_on) and t_plateau >= PSEUDO_PLATEAU_EPOCHS:
                t_pseudo_on = True
                print(f"  >> pseudo-labeling ENABLED at epoch {ep} (target-val plateaued)")
            if st['pseudo_used'] > 0:
                t_last_ps_acc = st['pseudo_acc']
                if t_last_ps_acc < PSEUDO_MIN_MONITOR_ACC:
                    t_pseudo_on = False
                    print(f"  >> pseudo-labeling AUTO-DISABLED: monitor acc "
                          f"{t_last_ps_acc:.3f} < {PSEUDO_MIN_MONITOR_ACC}")

        if ep % 5 == 0 or ep == 1:
            ps = f"{st['pseudo_acc']:.3f}" if st['pseudo_acc'] == st['pseudo_acc'] else "n/a"
            print(f"[T] Ep {ep:3d}/{TEACHER_EPOCHS} | cls={st['cls_loss']:.4f} "
                  f"coral={st['coral_loss']:.4f} mcc={st['mcc_loss']:.4f} im={st['im_loss']:+.4f} | "
                  f"TRAIN={st['train_acc']:.3f} SRC-VAL={sv:.3f} | TGT-VAL={tv:.3f} "
                  f"(sm={sm:.3f}) f1={tf1:.3f} | pseudo#={st['pseudo_used']} acc={ps} "
                  f"| {time.time()-t0:.0f}s")

        if t_stop.early_stop:
            print(f"[T] Early stop at epoch {ep}")
            break

    teacher.load_state_dict(t_best_state)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    torch.save(teacher.state_dict(), f"{MODELS_DIR}/teacher.pth")

    print(f"\nTeacher done in {time.time()-t0:.0f}s")
    print(f"  best RAW target-val      : {t_best_raw:.4f}")
    print(f"  best SMOOTHED target-val : {t_best_sm:.4f}   (v6 was 0.7753)")
    print("  This is the realistic ceiling for the student -- KD cannot exceed it.")
else:
    teacher, t_best_sm = None, 0.0
    print("USE_TEACHER_KD=False -- student will train with DA only.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 134MB/s] 


Teacher: mobilenet_v3_large, 2,974,835 params (training-time only, never exported)
  Teacher | 80 epochs | 224px | DA=IM-only(CORAL=False MMD=False MCC=False) | pseudo-labeling=OFF (v6 poisoned itself here)
[T] Ep   1/80 | cls=0.5035 coral=0.0000 mcc=0.0000 im=-0.0822 | TRAIN=0.897 SRC-VAL=0.993 | TGT-VAL=0.734 (sm=0.734) f1=0.743 | pseudo#=0 acc=n/a | 39s
[T] Ep   5/80 | cls=0.3488 coral=0.0000 mcc=0.0000 im=-0.3943 | TRAIN=0.986 SRC-VAL=0.998 | TGT-VAL=0.835 (sm=0.790) f1=0.829 | pseudo#=0 acc=n/a | 190s
[T] Ep  10/80 | cls=0.3326 coral=0.0000 mcc=0.0000 im=-0.4226 | TRAIN=0.990 SRC-VAL=0.996 | TGT-VAL=0.861 (sm=0.803) f1=0.855 | pseudo#=0 acc=n/a | 378s
[T] Ep  15/80 | cls=0.3431 coral=0.0000 mcc=0.0000 im=-0.4423 | TRAIN=0.988 SRC-VAL=0.996 | TGT-VAL=0.759 (sm=0.777) f1=0.765 | pseudo#=0 acc=n/a | 570s
[T] Ep  20/80 | cls=0.3245 coral=0.0000 mcc=0.0000 im=-0.4597 | TRAIN=0.994 SRC-VAL=0.998 | TGT-VAL=0.747 (sm=0.808) f1=0.725 | pseudo#=0 acc=n/a | 763s
[T] Ep  25/80 | cls=0.3142 co

In [ ]:
# ============================================================
# VIZ 5-1: Teacher training accuracy curve
# ============================================================
if USE_TEACHER_KD and 't_hist' in dir():
    ep_t = range(1, len(t_hist['train']) + 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ep_t, t_hist['train'], 'b-', lw=2, label='Train Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_title('Teacher — Training Accuracy', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_train_acc.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ep_t, t_hist['src_val'], 'b--', lw=2, label='Source Val Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_title('Teacher — Source Validation Accuracy', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_src_val_acc.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ep_t, t_hist['tgt_val'], 'r-', lw=2.5, label='Target Val Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_title('Teacher — Target Validation Accuracy', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_tgt_val_acc.png", dpi=150, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(ep_t, t_hist['im'], lw=2, color='#FB8C00', label='IM Loss')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Epoch'); ax.set_ylabel('IM Loss')
    ax.set_title('Teacher — IM Loss', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_im_loss.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Teacher history not available (loaded from checkpoint).")

In [ ]:
# ============================================================
# VIZ: Teacher — Training & Validation Accuracy
# ============================================================
if USE_TEACHER_KD and 't_hist' in dir() and t_hist:
    ep_t = range(1, len(t_hist['train']) + 1)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(ep_t, t_hist['train'], 'b-', lw=2, label='Training Accuracy')
    ax.plot(ep_t, t_hist['src_val'], 'g--', lw=2, label='Source Val Accuracy')
    ax.plot(ep_t, t_hist['tgt_val'], 'r-', lw=2.5, label='Target Val Accuracy')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Teacher — Training & Validation Accuracy', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_train_val_accuracy.png", dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# VIZ: Teacher — Training & Validation Loss
# ============================================================
if USE_TEACHER_KD and 't_hist' in dir() and t_hist and 'train_loss' in t_hist:
    ep_t = range(1, len(t_hist['train_loss']) + 1)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(ep_t, t_hist['train_loss'], 'b-', lw=2, label='Training Loss (CE)')
    ax.plot(ep_t, t_hist['val_loss'], 'r--', lw=2, label='Validation Loss')
    if any(v != 0 for v in t_hist['im']):
        ax.plot(ep_t, t_hist['im'], 'orange', lw=1.5, alpha=0.7, label='IM Loss')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('Teacher — Training & Validation Loss', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_train_val_loss.png", dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# VIZ: Teacher — Combined Training Curves (Accuracy + Loss)
# ============================================================
if USE_TEACHER_KD and 't_hist' in dir() and t_hist and 'train_loss' in t_hist:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Teacher — Training Curves", fontsize=16, fontweight='bold')
    ep_t = range(1, len(t_hist['train']) + 1)

    # Accuracy
    axes[0].plot(ep_t, t_hist['train'], 'b-', lw=2, label='Training')
    axes[0].plot(ep_t, t_hist['src_val'], 'g--', lw=2, label='Source Val')
    axes[0].plot(ep_t, t_hist['tgt_val'], 'r-', lw=2.5, label='Target Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Training & Validation Accuracy', fontweight='bold')
    axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

    # Loss
    axes[1].plot(ep_t, t_hist['train_loss'], 'b-', lw=2, label='Training Loss')
    axes[1].plot(ep_t, t_hist['val_loss'], 'r--', lw=2, label='Validation Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].set_title('Training & Validation Loss', fontweight='bold')
    axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/teacher_train_val_curves.png", dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# PART 5B: TEACHER SANITY + AdaBN CHECK
# FIX #13: AdaBN keep/discard is now decided on target-VAL, never target-TEST.
# The old version compared t_ab_acc vs t_te_acc (both computed on tgt_test_loader) and
# kept whichever scored higher ON THE TEST SET -- that's test-set peeking: reporting the
# best-of-2 test score instead of a single clean held-out measurement. Now the decision
# uses tgt_val_loader, and tgt_test_loader is touched exactly once, with the model the
# decision already settled on.
# ============================================================
if USE_TEACHER_KD:
    _, t_val_acc, _, _, _, _ = evaluate(teacher, tgt_val_loader, cls_criterion, DEVICE, 't')

    teacher_adabn = copy.deepcopy(teacher)
    adapt_bn(teacher_adabn, tgt_train_loader, DEVICE, which='t', n_pass=3)
    _, t_ab_val_acc, _, _, _, _ = evaluate(teacher_adabn, tgt_val_loader, cls_criterion, DEVICE, 't')

    print(f"Teacher target-VAL (no AdaBN) : acc={t_val_acc:.4f}")
    print(f"Teacher target-VAL (+ AdaBN)  : acc={t_ab_val_acc:.4f} (delta {t_ab_val_acc - t_val_acc:+.4f})")

    if t_ab_val_acc >= t_val_acc:
        teacher = teacher_adabn
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        print("  -> AdaBN version selected (decided on target-VAL, not test).")
    else:
        del teacher_adabn
        print("  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).")
    torch.cuda.empty_cache()

    # Test set touched exactly once, with whichever model the VAL-based decision above settled on.
    _, t_te_acc, t_te_f1, t_pred, t_lab, _ = evaluate(teacher, tgt_test_loader,
                                                      cls_criterion, DEVICE, 't')
    print(f"\nTeacher target-TEST (final, single measurement) : acc={t_te_acc:.4f} f1={t_te_f1:.4f}")
    print("Teacher target-test confusion (rows = actual):")
    cm = confusion_matrix(t_lab, t_pred)
    print(" " * 27 + "".join(f"{c[:13]:>15s}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        print(f"{c:<27s}" + "".join(f"{cm[i][j]:>15d}" for j in range(NUM_CLASSES)))


Teacher target-VAL (no AdaBN) : acc=0.8228
Teacher target-VAL (+ AdaBN)  : acc=0.8481 (delta +0.0253)
  -> AdaBN version selected (decided on target-VAL, not test).

Teacher target-TEST (final, single measurement) : acc=0.8228 f1=0.8287
Teacher target-test confusion (rows = actual):
                             Tomato_Septor  Potato_Early_  Corn_Leaf_Bli
Tomato_Septoria_Leaf_Spot               23              1              0
Potato_Early_Blight                      8             18              0
Corn_Leaf_Blight                         5              0             24


---
# Part 5C: Student

The deployed model: distilled from the teacher, adapted with CORAL + MMD + IM + MCC, optionally
self-trained. Every v6 failure mode is disarmed here:

- KD weight is **1.0 and constant**, not 3.0 and ramped, and the teacher is fed its own paired
  224px view of the same augmented image rather than a 96→224 upsample (FIX #4).
- **Feature distillation** through `kd_proj` supplements logit KD, which matters at a ~70x
  capacity gap. `kd_proj` is trained alongside the student and is not part of the exported graph.
- Alignment uses **IM only** for this run (`USE_CORAL=USE_MMD=USE_MCC=False`, `USE_IM=True`, Part 0) -- IM penalises the collapse that killed v6 (FIX #3) on its own; CORAL/MMD/MCC stay in the code (Part 3) and can be re-enabled by flipping those switches.
- Source and target share **one forward pass** for clean BN statistics (FIX #5).
- Pseudo-labeling waits for a target-val plateau, draws labels from the EMA model on the weak
  view, selects class-balanced top-k, and **switches itself off** below 0.85 monitor accuracy
  (FIX #1).

`COLLAPSE WARNING` fires whenever source-val lands within half a point of any `1 - class_share`
value. If that appears, stop the run — something is still erasing a class.

In [ ]:
# ============================================================
# PART 5C: STUDENT TRAINING
# ============================================================

def train_student(seed=SEED, epochs=EPOCHS, verbose=True, tag="student"):
    set_seed(seed)
    st_model = build_student().to(DEVICE)
    kd_proj = (nn.Linear(st_model.feature_dim, teacher.feature_dim).to(DEVICE)
               if (USE_TEACHER_KD and teacher is not None) else None)

    params = list(st_model.parameters()) + (list(kd_proj.parameters()) if kd_proj else [])
    opt = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, epochs, WARMUP_EPOCHS)
    stop = EarlyStopping(patience=PATIENCE, min_delta=1e-3)
    ema = EMA(st_model) if STUDENT_USE_PSEUDO else None

    hist = {k: [] for k in ['cls', 'coral', 'mmd', 'im', 'mcc', 'pseudo', 'kd', 'featkd',
                            'train_acc', 'src_val', 'tgt_val', 'tgt_f1', 'lam', 'lr',
                            'pseudo_used', 'pseudo_acc', 'val_loss']}
    raw, best_raw, best_sm = [], 0.0, 0.0
    best_state = copy.deepcopy(st_model.state_dict())
    pseudo_on, plateau, collapse_warned = False, 0, False

    shares = np.array([src_index.labels.count(i) for i in range(NUM_CLASSES)], float)
    shares /= shares.sum()
    collapse_vals = [1 - s for s in shares]
    t0 = time.time()

    if verbose:
        print("=" * 100)
        print(f"  Student [{tag}] seed={seed} | {epochs} epochs | KD="
              f"{'ON (lambda=%.1f const)' % LAMBDA_KD if kd_proj else 'OFF'} | "
              f"DA=IM-only(CORAL={USE_CORAL} MMD={USE_MMD} MCC={USE_MCC}) | "
              f"DA steps/epoch={min(len(src_train_loader), DA_TARGET_PASSES*len(tgt_train_loader))}")
        print("=" * 100)

    for ep in range(1, epochs + 1):
        st = train_da_epoch(st_model, src_train_loader, tgt_train_loader, cls_criterion,
                            opt, DEVICE, ep, epochs, CFG, which='s',
                            teacher=teacher if USE_TEACHER_KD else None,
                            kd_proj=kd_proj, ema=ema, pseudo_enabled=pseudo_on,
                            use_coral=USE_CORAL, use_mmd=USE_MMD, use_im=USE_IM, use_mcc=USE_MCC)
        sched.step()

        s_val_loss_, sv, _, _, _, _ = evaluate(st_model, src_val_loader, cls_criterion, DEVICE)
        _, tv, tf1, _, _, _ = evaluate(st_model, tgt_val_loader, cls_criterion, DEVICE)

        raw.append(tv)
        sm = float(np.mean(raw[-SMOOTH_WINDOW:]))
        for k, v in [('cls', st['cls_loss']), ('coral', st['coral_loss']), ('mmd', st['mmd_loss']),
                     ('im', st['im_loss']), ('mcc', st['mcc_loss']), ('pseudo', st['pseudo_loss']),
                     ('kd', st['kd_loss']), ('featkd', st['featkd_loss']),
                     ('train_acc', st['train_acc']), ('src_val', sv), ('tgt_val', tv),
                     ('tgt_f1', tf1), ('lam', st['lam']), ('lr', sched.get_last_lr()[0]),
                     ('pseudo_used', st['pseudo_used']), ('pseudo_acc', st['pseudo_acc']), ('val_loss', s_val_loss_)]:
            hist[k].append(v)

        best_raw = max(best_raw, tv)
        if sm > best_sm:
            best_sm = sm
            best_state = copy.deepcopy(st_model.state_dict())
        stop(sm, st_model)

        # ---- collapse detector: the v6 signature ----
        if (not collapse_warned) and ep > 12 and any(abs(sv - c) < 0.005 for c in collapse_vals):
            collapse_warned = True
            print(f"  !! COLLAPSE WARNING ep {ep}: src-val {sv:.4f} matches a "
                  f"'one class absorbed' value. This is the v6 signature -- stop and inspect.")

        # ---- pseudo-label control ----
        if STUDENT_USE_PSEUDO:
            plateau = plateau + 1 if (len(raw) > SMOOTH_WINDOW and sm <= best_sm) else 0
            if (not pseudo_on) and plateau >= PSEUDO_PLATEAU_EPOCHS:
                pseudo_on = True
                if verbose:
                    print(f"  >> pseudo-labeling ENABLED at epoch {ep}")
            if st['pseudo_used'] > 0 and st['pseudo_acc'] < PSEUDO_MIN_MONITOR_ACC:
                pseudo_on = False
                if verbose:
                    print(f"  >> pseudo-labeling AUTO-DISABLED (monitor acc "
                          f"{st['pseudo_acc']:.3f} < {PSEUDO_MIN_MONITOR_ACC})")

        if verbose and (ep % 5 == 0 or ep == 1):
            ps = f"{st['pseudo_acc']:.3f}" if st['pseudo_acc'] == st['pseudo_acc'] else "n/a"
            print(f"Ep {ep:3d}/{epochs} | cls={st['cls_loss']:.3f} coral={st['coral_loss']:.4f} "
                  f"mmd={st['mmd_loss']:.4f} im={st['im_loss']:+.3f} mcc={st['mcc_loss']:.3f} "
                  f"kd={st['kd_loss']:.3f} fkd={st['featkd_loss']:.3f} | "
                  f"TRAIN={st['train_acc']:.3f} SRC-VAL={sv:.3f} | TGT-VAL={tv:.3f} "
                  f"(sm={sm:.3f}) f1={tf1:.3f} | ps#={st['pseudo_used']} acc={ps} "
                  f"| {time.time()-t0:.0f}s")

        if stop.early_stop:
            if verbose:
                print(f"\nEarly stop at epoch {ep}")
            break

    st_model.load_state_dict(best_state)
    if verbose:
        print(f"\nDone in {time.time()-t0:.0f}s | best RAW tgt-val {best_raw:.4f} | "
              f"best SMOOTHED (selected) {best_sm:.4f}")
    return st_model, kd_proj, hist, best_sm


model, kd_proj, history, best_smoothed = train_student()
_da_parts = [n for n, u in [('CORAL', USE_CORAL), ('MMD', USE_MMD), ('IM', USE_IM), ('MCC', USE_MCC)] if u]
_method_str = '+'.join(_da_parts) + '+gatedPseudo+KD(logit+feature)'
torch.save({'model_state_dict': model.state_dict(), 'history': history,
            'config': {'img_size': IMG_SIZE, 'arch': STUDENT_ARCH, 'pretrained': STUDENT_PRETRAINED,
                       'classes': CLASSES, 'params': total_params,
                       'method': _method_str}},
           f"{MODELS_DIR}/student_final.pth")
print(f"Saved -> {MODELS_DIR}/student_final.pth")

  Student [student] seed=42 | 120 epochs | KD=ON (lambda=1.0 const) | DA=IM-only(CORAL=False MMD=False MCC=False) | DA steps/epoch=24
Ep   1/120 | cls=0.487 coral=0.0000 mmd=0.0000 im=-0.175 mcc=0.000 kd=0.475 fkd=0.001 | TRAIN=0.914 SRC-VAL=0.998 | TGT-VAL=0.873 (sm=0.873) f1=0.873 | ps#=0 acc=n/a | 52s
Ep   5/120 | cls=0.382 coral=0.0000 mmd=0.0000 im=-0.308 mcc=0.000 kd=0.297 fkd=0.001 | TRAIN=0.967 SRC-VAL=0.988 | TGT-VAL=0.797 (sm=0.787) f1=0.791 | ps#=0 acc=n/a | 262s
Ep  10/120 | cls=0.376 coral=0.0000 mmd=0.0000 im=-0.286 mcc=0.000 kd=0.288 fkd=0.001 | TRAIN=0.970 SRC-VAL=0.996 | TGT-VAL=0.797 (sm=0.770) f1=0.798 | ps#=0 acc=n/a | 523s
  >> pseudo-labeling ENABLED at epoch 13
Ep  15/120 | cls=0.365 coral=0.0000 mmd=0.0000 im=-0.321 mcc=0.000 kd=0.229 fkd=0.001 | TRAIN=0.976 SRC-VAL=1.000 | TGT-VAL=0.797 (sm=0.820) f1=0.797 | ps#=3 acc=1.000 | 787s
  >> pseudo-labeling AUTO-DISABLED (monitor acc 0.818 < 0.85)
  >> pseudo-labeling ENABLED at epoch 17
Ep  20/120 | cls=0.359 coral=

In [ ]:
# ============================================================
# VIZ 5C-1: Student CE loss
# ============================================================
ep_s = range(1, len(history['cls']) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['cls'], lw=2, color='#1565C0', label='CE Loss (source)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Student — Classification (CE) Loss', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_ce_loss.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-2: Student KD logit loss
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['kd'], lw=2, color='#6A1B9A', label='KD Logit Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Student — Knowledge Distillation (Logit) Loss', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_kd_loss.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-3: Student KD feature loss
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['featkd'], lw=2, color='#00695C', label='KD Feature Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Student — Knowledge Distillation (Feature) Loss', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_featkd_loss.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-4: Student IM loss
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['im'], lw=2, color='#FB8C00', label='IM Loss')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('IM Loss')
ax.set_title('Student — Information Maximization Loss\n(negative & falling = confident AND diverse)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_im_loss.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-5: Student train accuracy
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['train_acc'], 'b-', lw=2, label='Source Train Accuracy')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Student — Source Training Accuracy', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_train_acc.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-6: Student source validation accuracy
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['src_val'], 'b--', lw=2, label='Source Val Accuracy')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Student — Source Validation Accuracy', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_src_val_acc.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-7: Student target validation accuracy
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['tgt_val'], 'r-', lw=2.5, label='Target Val Accuracy')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Student — Target Validation Accuracy', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_tgt_val_acc.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-8: Student target validation F1 score
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['tgt_f1'], 'r--', lw=2, label='Target Val F1')
ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
ax.set_title('Student — Target Validation F1 Score', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_tgt_val_f1.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-9: Pseudo-labels used per epoch
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep_s, history['pseudo_used'], lw=2, color='#3949AB', label='# Pseudo-Labels Used')
ax.set_xlabel('Epoch'); ax.set_ylabel('Count')
ax.set_title('Student — Number of Pseudo-Labels Used per Epoch', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_pseudo_count.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-10: Pseudo-label accuracy per epoch
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
ps_acc_clean = [v if v == v else None for v in history['pseudo_acc']]
ax.plot(ep_s, ps_acc_clean, 'g--', lw=2, label='Pseudo-Label Accuracy')
ax.axhline(PSEUDO_MIN_MONITOR_ACC, color='r', ls=':', lw=2, label=f'Auto-off threshold ({PSEUDO_MIN_MONITOR_ACC})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Student — Pseudo-Label Accuracy', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_pseudo_acc.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-11: Learning rate schedule
# ============================================================
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ep_s, history['lr'], lw=2, color='#6D4C41', label='Learning Rate')
ax.set_xlabel('Epoch'); ax.set_ylabel('LR')
ax.set_title('Student — Learning Rate (Warmup + Cosine)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_lr.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 5C-12: Lambda ramp curve
# ============================================================
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ep_s, history['lam'], lw=2, color='#E65100', label='DA Lambda Ramp')
ax.set_xlabel('Epoch'); ax.set_ylabel('Lambda')
ax.set_title('Student — Domain Adaptation Lambda Ramp (sigmoid schedule)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_da_ramp.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ: Student — Training & Validation Accuracy
# ============================================================
ep_s = range(1, len(history['train_acc']) + 1)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ep_s, history['train_acc'], 'b-', lw=2, label='Training Accuracy')
ax.plot(ep_s, history['src_val'], 'g--', lw=2, label='Source Val Accuracy')
ax.plot(ep_s, history['tgt_val'], 'r-', lw=2.5, label='Target Val Accuracy')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Student — Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_train_val_accuracy.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ: Student — Training & Validation Loss
# ============================================================
ep_s = range(1, len(history['cls']) + 1)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ep_s, history['cls'], 'b-', lw=2, label='Training Loss (CE)')
ax.plot(ep_s, history['val_loss'], 'r--', lw=2, label='Validation Loss')
ax.plot(ep_s, history['kd'], 'orange', lw=1.5, alpha=0.7, label='KD Loss')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Student — Training & Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_train_val_loss.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ: Student — Combined Training Curves (Accuracy + Loss)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Student — Training Curves", fontsize=16, fontweight='bold')
ep_s = range(1, len(history['train_acc']) + 1)

# Accuracy
axes[0].plot(ep_s, history['train_acc'], 'b-', lw=2, label='Training')
axes[0].plot(ep_s, history['src_val'], 'g--', lw=2, label='Source Val')
axes[0].plot(ep_s, history['tgt_val'], 'r-', lw=2.5, label='Target Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training & Validation Accuracy', fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

# Loss
axes[1].plot(ep_s, history['cls'], 'b-', lw=2, label='Training Loss (CE)')
axes[1].plot(ep_s, history['val_loss'], 'r--', lw=2, label='Validation Loss')
axes[1].plot(ep_s, history['kd'], 'orange', lw=1.5, alpha=0.7, label='KD Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training & Validation Loss', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_train_val_curves.png", dpi=200, bbox_inches='tight')
plt.show()

---
# Part 6: AdaBN + evaluation

`adapt_bn` runs on a copy of the student using target-train images only, then both versions are
scored on the target test set and the better one is kept. This must happen **before** INT8
calibration in Part 10, otherwise quantization freezes the wrong normalization statistics into
the exported graph.

Every number is reported with a bootstrap 95% confidence interval. On 79 test images a single
image is worth 1.27 points and the interval is roughly ±9 points wide — two runs whose accuracies
fall inside each other's interval are not meaningfully different, and reporting a bare point
estimate would be misleading.

In [ ]:
# ============================================================
# PART 6: AdaBN + FULL EVALUATION
# ============================================================

def bootstrap_ci(labels, preds, n_boot=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    accs = np.array([(labels[i] == preds[i]).mean()
                     for i in (rng.randint(0, n, n) for _ in range(n_boot))])
    return accs.mean(), *np.percentile(accs, [2.5, 97.5])


# FIX #13: AdaBN keep/discard decided on target-VAL, never target-TEST (see Part 5B for
# why the old test-vs-test comparison was test-set peeking).
_, base_val_acc, _, _, _, _ = evaluate(model, tgt_val_loader, cls_criterion, DEVICE)

model_ab = copy.deepcopy(model)
adapt_bn(model_ab, tgt_train_loader, DEVICE, which='s', n_pass=3)
_, ab_val_acc, _, _, _, _ = evaluate(model_ab, tgt_val_loader, cls_criterion, DEVICE)

print(f"Student target-VAL (no AdaBN) : acc={base_val_acc:.4f}")
print(f"Student target-VAL (+ AdaBN)  : acc={ab_val_acc:.4f} (delta {ab_val_acc-base_val_acc:+.4f})")

if ab_val_acc >= base_val_acc:
    model = model_ab
    print("  -> AdaBN version selected (decided on target-VAL, not test).")
else:
    del model_ab
    print("  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).")

# Test set touched exactly once, with whichever model the VAL-based decision above settled on.
_, tgt_acc, tgt_f1, tgt_preds, tgt_labels, tgt_probs = evaluate(model, tgt_test_loader, cls_criterion, DEVICE)
print(f"\nStudent target-TEST (final, single measurement) : acc={tgt_acc:.4f} f1={tgt_f1:.4f}")
torch.save(model.state_dict(), f"{MODELS_DIR}/student_deploy.pth")

# FIX #12: clean (non-augmented) source-train loader and correct per-split predictions
_, src_train_acc, src_train_f1, sp_tr, sl_tr, _ = evaluate(model, src_train_eval_loader,
                                                           cls_criterion, DEVICE)
_, src_acc, src_f1, src_preds, src_labels, src_probs = evaluate(model, src_val_loader,
                                                                cls_criterion, DEVICE)
_, src_test_acc, src_test_f1, sp_te, sl_te, _ = evaluate(model, src_test_loader,
                                                         cls_criterion, DEVICE)

results = OrderedDict()
for name, (a, f, p, l) in [('Source Train (clean)', (src_train_acc, src_train_f1, sp_tr, sl_tr)),
                           ('Source Val',           (src_acc, src_f1, src_preds, src_labels)),
                           ('Source Test',          (src_test_acc, src_test_f1, sp_te, sl_te)),
                           ('Target Test',          (tgt_acc, tgt_f1, tgt_preds, tgt_labels))]:
    results[name] = {'accuracy': a, 'f1': f,
                     'precision': precision_score(l, p, average='weighted', zero_division=0),
                     'recall': recall_score(l, p, average='weighted', zero_division=0)}

print(f"\n{'='*78}")
print(f"{'Split':<24}{'Accuracy':<12}{'F1':<12}{'Precision':<12}{'Recall':<12}")
print("-" * 78)
for k, v in results.items():
    print(f"{k:<24}{v['accuracy']:<12.4f}{v['f1']:<12.4f}"
          f"{v['precision']:<12.4f}{v['recall']:<12.4f}")
print("=" * 78)
pd.DataFrame(results).T.to_csv(f"{TABLES_DIR}/performance_summary.csv")

bm, lo, hi = bootstrap_ci(tgt_labels, tgt_preds)
print(f"\nTarget test accuracy : {tgt_acc:.4f}")
print(f"Bootstrap 95% CI     : [{lo:.4f}, {hi:.4f}]  (n={len(tgt_labels)}, "
      f"1 image = {100/len(tgt_labels):.2f} points)")
print("Report the interval, not the point estimate.")

print(f"\nGeneralization gap (clean src-train -> tgt-test): {src_train_acc - tgt_acc:.4f}")
if USE_TEACHER_KD:
    print(f"\nCapacity chain on target test:")
    print(f"  Teacher ({TEACHER_BACKBONE}, 224px, {t_params/1e6:.1f}M) : {t_te_acc:.4f}")
    print(f"  Student ({IMG_SIZE}px, {total_params/1e6:.2f}M, DEPLOYED) : {tgt_acc:.4f}")
    print(f"  KD retention: {tgt_acc/max(t_te_acc,1e-9)*100:.1f}% of the teacher")

print("\nTarget-test classification report:")
print(classification_report(tgt_labels, tgt_preds, target_names=CLASSES, digits=4))

Student target-VAL (no AdaBN) : acc=0.8734
Student target-VAL (+ AdaBN)  : acc=0.8354 (delta -0.0380)
  -> Base (non-AdaBN) version kept (decided on target-VAL, not test).

Student target-TEST (final, single measurement) : acc=0.8101 f1=0.8108

Split                   Accuracy    F1          Precision   Recall      
------------------------------------------------------------------------------
Source Train (clean)    0.9989      0.9989      0.9989      0.9989      
Source Val              0.9982      0.9982      0.9982      0.9982      
Source Test             1.0000      1.0000      1.0000      1.0000      
Target Test             0.8101      0.8108      0.8158      0.8101      

Target test accuracy : 0.8101
Bootstrap 95% CI     : [0.7215, 0.8861]  (n=79, 1 image = 1.27 points)
Report the interval, not the point estimate.

Generalization gap (clean src-train -> tgt-test): 0.1887

Capacity chain on target test:
  Teacher (mobilenet_v3_large, 224px, 3.0M) : 0.8228
  Student (224px, 23.

In [ ]:
# ============================================================
# VIZ 6-1: Per-class accuracy for source val
# ============================================================
from sklearn.metrics import confusion_matrix as cm_fn

cm_src = cm_fn(src_labels, src_preds)
per_class_src = cm_src.diagonal() / cm_src.sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(NUM_CLASSES), per_class_src, color='#2196F3', alpha=0.85)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([c[:20] for c in CLASSES], rotation=15, fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy — Source Validation', fontweight='bold')
ax.set_ylim(0, 1.1)
for i, v in enumerate(per_class_src):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/per_class_acc_source.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 6-2: Per-class accuracy for target test
# ============================================================
cm_tgt = cm_fn(tgt_labels, tgt_preds)
per_class_tgt = cm_tgt.diagonal() / cm_tgt.sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(NUM_CLASSES), per_class_tgt, color='#FF5722', alpha=0.85)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([c[:20] for c in CLASSES], rotation=15, fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy — Target Test', fontweight='bold')
ax.set_ylim(0, 1.1)
for i, v in enumerate(per_class_tgt):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/per_class_acc_target.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PART 6B: CONFUSION MATRICES + ROC + TRAINING CURVES
# ============================================================

def plot_cm(preds, labels, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
                xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
    ax[0].set_title('Raw'); ax[0].set_ylabel('Actual'); ax[0].set_xlabel('Predicted')
    sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt='.3f', cmap='Greens', ax=ax[1],
                xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
    ax[1].set_title('Normalized'); ax[1].set_ylabel('Actual'); ax[1].set_xlabel('Predicted')
    plt.tight_layout(); plt.savefig(f"{FIGURES_DIR}/{fname}", dpi=200, bbox_inches='tight'); plt.show()


plot_cm(src_preds, src_labels, "Source (validation)", "cm_source.png")
plot_cm(tgt_preds, tgt_labels, "Target test -- deployed student", "cm_target.png")


def plot_roc(probs, labels, title, fname):
    fig, ax = plt.subplots(figsize=(8, 6))
    lb = label_binarize(labels, classes=range(NUM_CLASSES))
    aucs = {}
    for i, c in enumerate(['#E53935', '#1E88E5', '#43A047']):
        fpr, tpr, _ = roc_curve(lb[:, i], probs[:, i])
        aucs[CLASSES[i]] = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=c, lw=2.2, label=f'{CLASSES[i][:18]} (AUC={aucs[CLASSES[i]]:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=.3)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{title}\nMacro AUC = {np.mean(list(aucs.values())):.4f}', fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.savefig(f"{FIGURES_DIR}/{fname}", dpi=200, bbox_inches='tight'); plt.show()
    return float(np.mean(list(aucs.values())))


src_macro = plot_roc(src_probs, src_labels, "Source ROC", "roc_source.png")
tgt_macro = plot_roc(tgt_probs, tgt_labels, "Target ROC", "roc_target.png")
print(f"Macro AUC  source={src_macro:.4f}  target={tgt_macro:.4f}")

# ---- training dynamics ----
ep = range(1, len(history['cls']) + 1)
fig, ax = plt.subplots(3, 2, figsize=(16, 15))
fig.suptitle("Student training dynamics (v7)", fontsize=15, fontweight='bold')

ax[0][0].plot(ep, history['cls'], lw=2, label='CE (source)')
ax[0][0].plot(ep, history['kd'], lw=2, label='KD (logit)')
ax[0][0].plot(ep, history['featkd'], lw=2, label='KD (feature)')
ax[0][0].set_title('Classification & distillation', fontweight='bold'); ax[0][0].legend()

ax[0][1].plot(ep, history['coral'], lw=2, label=f'CORAL (USE_CORAL={USE_CORAL})')
ax[0][1].plot(ep, history['mmd'], lw=2, label=f'MMD (USE_MMD={USE_MMD})')
ax[0][1].plot(ep, history['mcc'], lw=2, label=f'MCC (USE_MCC={USE_MCC})')
ax[0][1].set_title('Alignment losses (flat 0 = disabled this run, IM-only)', fontweight='bold')
ax[0][1].legend()

ax[1][0].plot(ep, history['train_acc'], 'b-', lw=2, label='Source train')
ax[1][0].plot(ep, history['src_val'], 'b--', lw=2, label='Source val')
ax[1][0].plot(ep, history['tgt_val'], 'r-', lw=2.5, label='TARGET val')
shares = np.array([src_index.labels.count(i) for i in range(NUM_CLASSES)], float)
shares /= shares.sum()
for s in shares:
    ax[1][0].axhline(1 - s, color='gray', ls=':', alpha=.6)
ax[1][0].set_title('Accuracy (dotted grey = collapse levels)', fontweight='bold'); ax[1][0].legend()

ax[1][1].plot(ep, history['im'], lw=2, color='#FB8C00')
ax[1][1].axhline(0, color='k', lw=.8)
ax[1][1].set_title('IM loss = H(individual) - H(batch mean)\nnegative & falling = confident AND diverse',
                   fontweight='bold')

ax[2][0].plot(ep, history['pseudo_used'], lw=2, color='#3949AB', label='# pseudo-labels')
a2 = ax[2][0].twinx()
a2.plot(ep, [v if v == v else np.nan for v in history['pseudo_acc']], 'g--', lw=2,
        label='pseudo-label accuracy')
a2.axhline(PSEUDO_MIN_MONITOR_ACC, color='r', ls=':', label='auto-off threshold')
a2.set_ylim(0, 1.05)
ax[2][0].set_title('Self-training volume & quality', fontweight='bold')
ax[2][0].legend(loc='upper left'); a2.legend(loc='lower right')

ax[2][1].plot(ep, history['lr'], lw=2, color='#6D4C41')
ax[2][1].set_title('LR (warmup + cosine)', fontweight='bold')
for a in ax.ravel():
    a.grid(alpha=.3); a.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig(f"{FIGURES_DIR}/training_dynamics.png", dpi=200,
                                bbox_inches='tight'); plt.show()

# ---- degenerate-alignment monitor (FIX #9) -- only meaningful when CORAL is active ----
if USE_CORAL:
    c0, c1 = np.mean(history['coral'][:5]), np.mean(history['coral'][-5:])
    a0, a1 = np.mean(history['tgt_val'][:5]), np.mean(history['tgt_val'][-5:])
    print(f"\nDegenerate-alignment check: CORAL {c0:.5f} -> {c1:.5f} | "
          f"target-val {a0:.4f} -> {a1:.4f}")
    if c1 < 0.1 * c0 and a1 < a0 + 0.02:
        print("  WARNING: CORAL collapsed while target accuracy stood still -- the model is likely")
        print("  aligning domains by discarding class structure. Lower LAMBDA_CORAL_MAX further.")
    else:
        print("  OK: alignment improved alongside target accuracy.")
else:
    a0, a1 = np.mean(history['tgt_val'][:5]), np.mean(history['tgt_val'][-5:])
    print(f"\nIM-only run (USE_CORAL=False): CORAL check skipped. "
          f"target-val {a0:.4f} -> {a1:.4f}")

Macro AUC  source=1.0000  target=0.9213

IM-only run (USE_CORAL=False): CORAL check skipped. target-val 0.7873 -> 0.8304


In [ ]:
# ============================================================
# VIZ 6B-1: Source confusion matrix (raw) — individual
# ============================================================
cm = confusion_matrix(src_labels, src_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
ax.set_title('Source Val — Confusion Matrix (Raw)', fontweight='bold')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cm_source_raw.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 6B-2: Source confusion matrix (normalized) — individual
# ============================================================
cm = confusion_matrix(src_labels, src_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt='.3f', cmap='Greens', ax=ax,
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
ax.set_title('Source Val — Confusion Matrix (Normalized)', fontweight='bold')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cm_source_normalized.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 6B-3: Target confusion matrix (raw) — individual
# ============================================================
cm = confusion_matrix(tgt_labels, tgt_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
ax.set_title('Target Test — Confusion Matrix (Raw)', fontweight='bold')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cm_target_raw.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 6B-4: Target confusion matrix (normalized) — individual
# ============================================================
cm = confusion_matrix(tgt_labels, tgt_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt='.3f', cmap='Greens', ax=ax,
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
ax.set_title('Target Test — Confusion Matrix (Normalized)', fontweight='bold')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/cm_target_normalized.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 6B-5: Source ROC per class — individual figures
# ============================================================
lb_src = label_binarize(src_labels, classes=range(NUM_CLASSES))
for ci in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(lb_src[:, ci], src_probs[:, ci])
    auc_val = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr, tpr, lw=2.5, color=['#E53935', '#1E88E5', '#43A047'][ci])
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'Source ROC — {CLASSES[ci]} (AUC={auc_val:.4f})', fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/roc_source_{CLASSES[ci]}.png", dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# VIZ 6B-6: Target ROC per class — individual figures
# ============================================================
lb_tgt = label_binarize(tgt_labels, classes=range(NUM_CLASSES))
for ci in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(lb_tgt[:, ci], tgt_probs[:, ci])
    auc_val = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr, tpr, lw=2.5, color=['#E53935', '#1E88E5', '#43A047'][ci])
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'Target ROC — {CLASSES[ci]} (AUC={auc_val:.4f})', fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/roc_target_{CLASSES[ci]}.png", dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# VIZ 6B-7: Prediction confidence histogram — target test
# ============================================================
max_confs = np.max(tgt_probs, axis=1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(max_confs, bins=30, color='#FF5722', alpha=0.8, edgecolor='white')
ax.set_xlabel('Max Prediction Confidence')
ax.set_ylabel('Count')
ax.set_title('Target Test — Prediction Confidence Distribution', fontweight='bold')
ax.axvline(np.mean(max_confs), color='k', ls='--', lw=2, label=f'Mean={np.mean(max_confs):.3f}')
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/confidence_hist_target.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 6B-8: Confidence for correct vs incorrect predictions
# ============================================================
correct_mask = (tgt_preds == tgt_labels)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(max_confs[correct_mask], bins=20, alpha=0.7, color='#4CAF50', label='Correct', edgecolor='white')
ax.hist(max_confs[~correct_mask], bins=20, alpha=0.7, color='#F44336', label='Incorrect', edgecolor='white')
ax.set_xlabel('Max Prediction Confidence')
ax.set_ylabel('Count')
ax.set_title('Confidence: Correct vs Incorrect Predictions (Target)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/confidence_correct_vs_incorrect.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ: Teacher — Source (Combined CM + ROC)
# ============================================================
_, t_src_acc, t_src_f1, t_src_preds, t_src_labels, t_src_probs = evaluate(
    teacher, src_val_loader, cls_criterion, DEVICE, which='t')

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle("Teacher -- Source", fontsize=16, fontweight='bold')

cm = confusion_matrix(t_src_labels, t_src_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

lb = label_binarize(t_src_labels, classes=range(NUM_CLASSES))
aucs = {}
for i, clr in enumerate(['#E53935', '#1E88E5', '#43A047']):
    fpr, tpr, _ = roc_curve(lb[:, i], t_src_probs[:, i])
    aucs[CLASSES[i]] = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=clr, lw=2.2,
                 label=f'{CLASSES[i][:18]} (AUC={aucs[CLASSES[i]]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
t_src_macro_auc = np.mean(list(aucs.values()))
axes[1].set_title(f'ROC (Macro AUC = {t_src_macro_auc:.4f})', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/teacher_source_cm_roc.png", dpi=200, bbox_inches='tight')
plt.show()
print(f"Teacher Source — Acc: {t_src_acc:.4f} | F1: {t_src_f1:.4f} | Macro AUC: {t_src_macro_auc:.4f}")

Teacher Source — Acc: 0.9894 | F1: 0.9893 | Macro AUC: 1.0000


In [ ]:
# ============================================================
# VIZ: Teacher — Target (Combined CM + ROC)
# ============================================================
_, t_tgt_acc, t_tgt_f1, t_tgt_preds, t_tgt_labels, t_tgt_probs = evaluate(
    teacher, tgt_test_loader, cls_criterion, DEVICE, which='t')

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle("Teacher -- Target", fontsize=16, fontweight='bold')

cm = confusion_matrix(t_tgt_labels, t_tgt_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

lb = label_binarize(t_tgt_labels, classes=range(NUM_CLASSES))
aucs = {}
for i, clr in enumerate(['#E53935', '#1E88E5', '#43A047']):
    fpr, tpr, _ = roc_curve(lb[:, i], t_tgt_probs[:, i])
    aucs[CLASSES[i]] = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=clr, lw=2.2,
                 label=f'{CLASSES[i][:18]} (AUC={aucs[CLASSES[i]]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
t_tgt_macro_auc = np.mean(list(aucs.values()))
axes[1].set_title(f'ROC (Macro AUC = {t_tgt_macro_auc:.4f})', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/teacher_target_cm_roc.png", dpi=200, bbox_inches='tight')
plt.show()
print(f"Teacher Target — Acc: {t_tgt_acc:.4f} | F1: {t_tgt_f1:.4f} | Macro AUC: {t_tgt_macro_auc:.4f}")

Teacher Target — Acc: 0.8228 | F1: 0.8287 | Macro AUC: 0.9097


In [ ]:
# ============================================================
# VIZ: Student — Source (Combined CM + ROC)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle("Student -- Source", fontsize=16, fontweight='bold')

cm = confusion_matrix(src_labels, src_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

lb = label_binarize(src_labels, classes=range(NUM_CLASSES))
aucs = {}
for i, clr in enumerate(['#E53935', '#1E88E5', '#43A047']):
    fpr, tpr, _ = roc_curve(lb[:, i], src_probs[:, i])
    aucs[CLASSES[i]] = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=clr, lw=2.2,
                 label=f'{CLASSES[i][:18]} (AUC={aucs[CLASSES[i]]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
s_src_macro_auc = np.mean(list(aucs.values()))
axes[1].set_title(f'ROC (Macro AUC = {s_src_macro_auc:.4f})', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_source_cm_roc.png", dpi=200, bbox_inches='tight')
plt.show()
print(f"Student Source — Acc: {src_acc:.4f} | F1: {src_f1:.4f} | Macro AUC: {s_src_macro_auc:.4f}")

Student Source — Acc: 0.9982 | F1: 0.9982 | Macro AUC: 1.0000


In [ ]:
# ============================================================
# VIZ: Student — Target (Combined CM + ROC)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle("Student -- Target", fontsize=16, fontweight='bold')

cm = confusion_matrix(tgt_labels, tgt_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[c[:14] for c in CLASSES], yticklabels=[c[:14] for c in CLASSES])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

lb = label_binarize(tgt_labels, classes=range(NUM_CLASSES))
aucs = {}
for i, clr in enumerate(['#E53935', '#1E88E5', '#43A047']):
    fpr, tpr, _ = roc_curve(lb[:, i], tgt_probs[:, i])
    aucs[CLASSES[i]] = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=clr, lw=2.2,
                 label=f'{CLASSES[i][:18]} (AUC={aucs[CLASSES[i]]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
s_tgt_macro_auc = np.mean(list(aucs.values()))
axes[1].set_title(f'ROC (Macro AUC = {s_tgt_macro_auc:.4f})', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/student_target_cm_roc.png", dpi=200, bbox_inches='tight')
plt.show()
print(f"Student Target — Acc: {tgt_acc:.4f} | F1: {tgt_f1:.4f} | Macro AUC: {s_tgt_macro_auc:.4f}")

Student Target — Acc: 0.8101 | F1: 0.8108 | Macro AUC: 0.9213


In [ ]:
# ============================================================
# PART 8: FEATURE SPACE (t-SNE / PCA) + GRAD-CAM
# ============================================================
@torch.no_grad()
def collect_feats(m, loader, device, max_n=800):
    m.eval(); F_, L_ = [], []
    for xs, xt, xw_s, xw_t, y, d in loader:
        F_.append(m.extract_features(xs.to(device)).cpu().numpy()); L_.extend(y.numpy())
        if len(L_) >= max_n:
            break
    return np.vstack(F_)[:max_n], np.array(L_[:max_n])


fs_, ls_ = collect_feats(model, src_val_loader, DEVICE)
ft_, lt_ = collect_feats(model, tgt_test_loader, DEVICE)
X = np.vstack([fs_, ft_])
dom = np.array([0] * len(fs_) + [1] * len(ft_))
lab = np.concatenate([ls_, lt_])

for reducer, nm in [(TSNE(2, random_state=SEED, perplexity=30, init='pca'), 'tsne'),
                    (PCA(2), 'pca')]:
    Z = reducer.fit_transform(X)
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Feature space ({nm.upper()})", fontsize=14, fontweight='bold')
    ax[0].scatter(Z[dom == 0, 0], Z[dom == 0, 1], c='#2196F3', s=12, alpha=.5, label='Source')
    ax[0].scatter(Z[dom == 1, 0], Z[dom == 1, 1], c='#F44336', s=12, alpha=.6, label='Target')
    ax[0].set_title("By domain (mixed = aligned)"); ax[0].legend()
    for i, c in enumerate(['#E53935', '#1E88E5', '#43A047']):
        ax[1].scatter(Z[lab == i, 0], Z[lab == i, 1], c=c, s=12, alpha=.5, label=CLASSES[i][:16])
    ax[1].set_title("By class (separated = discriminative)"); ax[1].legend()
    for a in ax:
        a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{FIGURES_DIR}/features_{nm}.png", dpi=200,
                                    bbox_inches='tight'); plt.show()

print("Read both panels together: domains mixed AND classes separated = healthy adaptation.")
print("Domains mixed but classes merged = the degenerate alignment that produced v6's 0.730.")

# ---------------- Grad-CAM ----------------
from scipy.ndimage import zoom as scipy_zoom


class GradCAM:
    def __init__(self, m, layer):
        self.m = m; self.g = None; self.a = None
        layer.register_forward_hook(lambda mod, i, o: setattr(self, 'a', o.detach()))
        layer.register_full_backward_hook(lambda mod, gi, go: setattr(self, 'g', go[0].detach()))

    def generate(self, x, idx=None):
        self.m.zero_grad()
        out = self.m(x)
        idx = out.argmax(1).item() if idx is None else idx
        out[0, idx].backward()
        w = self.g[0].mean((1, 2), keepdim=True)
        cam = F.relu((w * self.a[0]).sum(0))
        cam = cam / cam.max() if cam.max() > 0 else cam
        return cam.cpu().numpy(), idx, F.softmax(out, 1)[0].detach().cpu().numpy()


gc = GradCAM(model, model.features[-1][-1])
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(15, 4 * NUM_CLASSES))
fig.suptitle("Grad-CAM on target (PlantDoc) -- attention should sit on lesions, not background",
             fontsize=14, fontweight='bold')
done = [0] * NUM_CLASSES
for xs, xt, xw_s, xw_t, y, d in tgt_test_loader:
    for i in range(len(xs)):
        c = int(y[i])
        if done[c]:
            continue
        cam, pred, pr = gc.generate(xs[i:i+1].to(DEVICE).requires_grad_(True))
        orig = np.clip(xs[i].permute(1, 2, 0).numpy() * .5 + .5, 0, 1)
        up = np.clip(scipy_zoom(cam, IMG_SIZE / cam.shape[0], order=1), 0, 1)
        axes[c][0].imshow(orig); axes[c][0].set_title(f"GT: {CLASSES[c][:18]}", fontsize=9)
        axes[c][1].imshow(up, cmap='jet'); axes[c][1].set_title("Grad-CAM", fontsize=9)
        axes[c][2].imshow(orig); axes[c][2].imshow(up, cmap='jet', alpha=.5)
        axes[c][2].set_title("Overlay", fontsize=9)
        axes[c][3].barh(range(NUM_CLASSES), pr, color=['#E53935', '#1E88E5', '#43A047'])
        axes[c][3].set_yticks(range(NUM_CLASSES))
        axes[c][3].set_yticklabels([cc[:10] for cc in CLASSES], fontsize=7)
        axes[c][3].set_xlim(0, 1); axes[c][3].set_title(f"Pred: {CLASSES[pred][:14]}", fontsize=9)
        for k in range(3):
            axes[c][k].axis('off')
        done[c] = 1
    if all(done):
        break
plt.tight_layout(); plt.savefig(f"{FIGURES_DIR}/gradcam.png", dpi=200, bbox_inches='tight'); plt.show()

Read both panels together: domains mixed AND classes separated = healthy adaptation.
Domains mixed but classes merged = the degenerate alignment that produced v6's 0.730.


In [ ]:
# ============================================================
# VIZ 8-1: t-SNE by domain — individual figure
# ============================================================
Z_tsne = TSNE(2, random_state=SEED, perplexity=30, init='pca').fit_transform(X)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(Z_tsne[dom==0, 0], Z_tsne[dom==0, 1], c='#2196F3', s=15, alpha=.5, label='Source')
ax.scatter(Z_tsne[dom==1, 0], Z_tsne[dom==1, 1], c='#F44336', s=15, alpha=.6, label='Target')
ax.set_title('t-SNE — By Domain (mixed = aligned)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/tsne_domain.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 8-2: t-SNE by class — individual figure
# ============================================================
fig, ax = plt.subplots(figsize=(8, 8))
for i, c in enumerate(['#E53935', '#1E88E5', '#43A047']):
    ax.scatter(Z_tsne[lab==i, 0], Z_tsne[lab==i, 1], c=c, s=15, alpha=.5, label=CLASSES[i][:16])
ax.set_title('t-SNE — By Class (separated = discriminative)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/tsne_class.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 8-3: PCA by domain — individual figure
# ============================================================
Z_pca = PCA(2).fit_transform(X)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(Z_pca[dom==0, 0], Z_pca[dom==0, 1], c='#2196F3', s=15, alpha=.5, label='Source')
ax.scatter(Z_pca[dom==1, 0], Z_pca[dom==1, 1], c='#F44336', s=15, alpha=.6, label='Target')
ax.set_title('PCA — By Domain (mixed = aligned)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/pca_domain.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 8-4: PCA by class — individual figure
# ============================================================
fig, ax = plt.subplots(figsize=(8, 8))
for i, c in enumerate(['#E53935', '#1E88E5', '#43A047']):
    ax.scatter(Z_pca[lab==i, 0], Z_pca[lab==i, 1], c=c, s=15, alpha=.5, label=CLASSES[i][:16])
ax.set_title('PCA — By Class (separated = discriminative)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/pca_class.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VIZ 8-5: Grad-CAM per class — individual image, overlay, and prob bar
# ============================================================
gc2 = GradCAM(model, model.features[-1][-1])
done2 = [0] * NUM_CLASSES
for xs, xt, xw_s, xw_t, y, d in tgt_test_loader:
    for i in range(len(xs)):
        c = int(y[i])
        if done2[c]:
            continue
        cam, pred, pr = gc2.generate(xs[i:i+1].to(DEVICE).requires_grad_(True))
        orig = np.clip(xs[i].permute(1, 2, 0).numpy() * .5 + .5, 0, 1)
        up = np.clip(scipy_zoom(cam, IMG_SIZE / cam.shape[0], order=1), 0, 1)

        # Original
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(orig); ax.set_title(f"Original — GT: {CLASSES[c][:20]}", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/gradcam_original_{CLASSES[c]}.png", dpi=200, bbox_inches='tight')
        plt.show()

        # Heatmap
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(up, cmap='jet'); ax.set_title(f"Grad-CAM Heatmap — {CLASSES[c][:20]}", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/gradcam_heatmap_{CLASSES[c]}.png", dpi=200, bbox_inches='tight')
        plt.show()

        # Overlay
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(orig); ax.imshow(up, cmap='jet', alpha=0.5)
        ax.set_title(f"Grad-CAM Overlay — {CLASSES[c][:20]}", fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/gradcam_overlay_{CLASSES[c]}.png", dpi=200, bbox_inches='tight')
        plt.show()

        # Prediction probabilities
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.barh(range(NUM_CLASSES), pr, color=['#E53935', '#1E88E5', '#43A047'])
        ax.set_yticks(range(NUM_CLASSES))
        ax.set_yticklabels([cc[:18] for cc in CLASSES])
        ax.set_xlim(0, 1); ax.set_xlabel('Probability')
        ax.set_title(f"Prediction — Pred: {CLASSES[pred][:20]}", fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/gradcam_probs_{CLASSES[c]}.png", dpi=200, bbox_inches='tight')
        plt.show()

        done2[c] = 1
    if all(done2):
        break

---
# Part 10: Export — ONNX → TFLite INT8 → ESP32-S3 (FIX #13)

v6 evaluated `tflite_fp32_path` and `tflite_int8_path` without ever defining them, so this section
raised `NameError` before producing anything. The actual `onnx2tf` conversion is included here.

Two invariants the calibration must respect:

1. **AdaBN first.** Part 6 already swapped in the target-adapted BN statistics; INT8 calibration
   freezes whatever statistics are present at export time.
2. **Same preprocessing as training** — `Resize(224) → ToTensor → Normalize(0.5, 0.5)` (the ResNet-50 wrapper converts to ImageNet normalisation internally), giving
   roughly [-1, 1] — and calibration samples drawn from **both** domains. Target samples come from
   the target *validation* split; the test split stays untouched until the accuracy check.

In [ ]:
# Install BEFORE the ONNX export below -- torch.onnx.export needs onnx/onnxscript present.
!pip install -q onnx onnxscript "onnx2tf==2.5.2" onnx-graphsurgeon sng4onnx onnxsim tensorflow tflite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.

In [ ]:
# ============================================================
# PART 10A: DEPLOY WRAPPER + ONNX  (FIX #15: static batch export)
# ============================================================
# FIX: v7 exported with dynamic_axes on batch. onnx2tf then inserts Shape/Gather/Equal
# ops to reconstruct the dynamic batch dim at runtime -- and EQUAL has no kernel in
# onnx2tf's strict full-integer quantization builder, so INT8 export crashed.
# ESP32-S3 inference is always batch=1, so there is no reason to keep the batch axis
# dynamic. Static export removes the Shape/Equal ops entirely.

deploy_model = copy.deepcopy(model).cpu().eval()
deploy_params = sum(p.numel() for p in deploy_model.parameters())
print(f"Deploy params: {deploy_params:,} | FP32 {deploy_params*4/1024:.1f} KB | "
      f"INT8 ~{deploy_params/1024:.1f} KB")

onnx_path = f"{MODELS_DIR}/resnet50_student.onnx"
torch.onnx.export(deploy_model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), onnx_path,
                  input_names=['input'], output_names=['output'], opset_version=16,
                  dynamo=False)   # <-- no dynamic_axes: batch is fixed at 1, matching deployment
print(f"ONNX: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")

try:
    import onnx
    onnx.checker.check_model(onnx.load(onnx_path))
    print("ONNX check passed")
except Exception as e:
    print("ONNX check:", e)

# Sanity check: confirm no dynamic dims remain on the graph input
m = onnx.load(onnx_path)
in_shape = [d.dim_value if d.dim_value > 0 else f"dynamic:{d.dim_param}"
            for d in m.graph.input[0].type.tensor_type.shape.dim]
print(f"ONNX input shape: {in_shape}  (all dims should be fixed integers, no 'dynamic:*')")

# ---- calibration set: same preprocessing as training, BOTH domains, test set untouched ----
cal = []
for loader, cap in [(src_val_loader, 400), (tgt_val_loader, 400)]:
    n = 0
    for xs, xt, xw_s, xw_t, y, d in loader:
        for im_ in xs:
            cal.append(im_.unsqueeze(0).numpy().astype(np.float32)); n += 1
        if n >= cap:
            break
np.random.shuffle(cal)
cal_arr = np.concatenate(cal, 0)
cal_nhwc = np.transpose(cal_arr, (0, 2, 3, 1))          # onnx2tf calibrates in NHWC
np.save(f"{MODELS_DIR}/calibration_nhwc.npy", cal_nhwc)
print(f"Calibration: {cal_nhwc.shape}, range [{cal_nhwc.min():.2f}, {cal_nhwc.max():.2f}] "
      f"(expect ~[-1, 1])")
assert -1.6 < cal_nhwc.min() < 0 < cal_nhwc.max() < 1.6, "preprocessing mismatch!"
print("Preprocessing consistency verified.")

Deploy params: 23,514,179 | FP32 91852.3 KB | INT8 ~22963.1 KB
ONNX: /kaggle/working/resnet50_student/models/resnet50_student.onnx (91779.8 KB)
ONNX check passed
ONNX input shape: [1, 3, 224, 224]  (all dims should be fixed integers, no 'dynamic:*')
Calibration: (511, 224, 224, 3), range [-1.00, 1.00] (expect ~[-1, 1])
Preprocessing consistency verified.


In [ ]:
# ============================================================
# PART 10B: NATIVE TF/KERAS REBUILD OF ResNet-50 -> TFLITE (FP32 + full INT8)
# ============================================================
# Same strategy as the original 10B (skip onnx2tf, rebuild natively in Keras, copy PyTorch weights,
# verify accuracy, quantize with the standard TFLiteConverter) -- but for torchvision ResNet-50 (v1.5:
# the stride sits on the 3x3 conv). The Keras model takes RAW [0,1] NHWC pixels and applies the ImageNet
# normalisation internally, so Part 10C runs unchanged.
import gc
import tensorflow as tf
try:
    tf.config.set_visible_devices([], 'GPU')     # keep TF on CPU; PyTorch owns the GPU
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()

def mk_eval_loader_safe(ds, bs):
    return DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)
tgt_test_loader = mk_eval_loader_safe(tgt_test_ds, BATCH_SIZE)

L = tf.keras.layers
_MEAN    = np.array(list(IMAGENET_MEAN), np.float32).reshape(1, 1, 1, 3)
_INV_STD = (1.0 / np.array(list(IMAGENET_STD), np.float32)).reshape(1, 1, 1, 3)
RESNET50_CFG = [(3, 64, 1), (4, 128, 2), (6, 256, 2), (3, 512, 2)]     # (blocks, mid channels, first stride)

def _bn(x, name):
    return L.BatchNormalization(epsilon=1e-5, name=name)(x)

def bottleneck_tf(x, mid, stride, has_down, name):
    out_ch = mid * 4
    y = L.Conv2D(mid, 1, use_bias=False, name=f"{name}_conv1")(x)
    y = L.ReLU(name=f"{name}_relu1")(_bn(y, f"{name}_bn1"))
    if stride == 1:
        y = L.Conv2D(mid, 3, padding='same', use_bias=False, name=f"{name}_conv2")(y)
    else:
        y = L.ZeroPadding2D(1, name=f"{name}_pad2")(y)
        y = L.Conv2D(mid, 3, strides=stride, padding='valid', use_bias=False, name=f"{name}_conv2")(y)
    y = L.ReLU(name=f"{name}_relu2")(_bn(y, f"{name}_bn2"))
    y = L.Conv2D(out_ch, 1, use_bias=False, name=f"{name}_conv3")(y)
    y = _bn(y, f"{name}_bn3")
    sc = x
    if has_down:
        sc = L.Conv2D(out_ch, 1, strides=stride, use_bias=False, name=f"{name}_down_conv")(x)
        sc = _bn(sc, f"{name}_down_bn")
    return L.ReLU(name=f"{name}_out_relu")(L.Add(name=f"{name}_add")([sc, y]))

def build_resnet50_tf(img_size, num_classes):
    inp = tf.keras.Input(shape=(img_size, img_size, 3), batch_size=1, name="input")
    x = L.Lambda(lambda t: (t - _MEAN) * _INV_STD, name="input_norm")(inp)      # raw [0,1] -> ImageNet-normalised
    x = L.ZeroPadding2D(3, name="stem_pad")(x)
    x = L.Conv2D(64, 7, strides=2, padding='valid', use_bias=False, name="stem_conv")(x)
    x = L.ReLU(name="stem_relu")(_bn(x, "stem_bn"))
    x = L.ZeroPadding2D(1, name="pool_pad")(x)                                  # post-ReLU >= 0, so zero-pad == -inf-pad
    x = L.MaxPooling2D(3, strides=2, padding='valid', name="stem_pool")(x)
    for li, (n_blocks, mid, s) in enumerate(RESNET50_CFG, 1):
        for bi in range(n_blocks):
            x = bottleneck_tf(x, mid, s if bi == 0 else 1, has_down=(bi == 0), name=f"l{li}b{bi}")
    x = L.GlobalAveragePooling2D(name="gap")(x)
    out = L.Dense(num_classes, name="cls_fc")(x)
    return tf.keras.Model(inp, out, name="ResNet50Student_Deploy")

print("1. Building TF/Keras ResNet-50...")
tf_model = build_resnet50_tf(IMG_SIZE, NUM_CLASSES)
print(f"   TF model params: {tf_model.count_params():,}   (PyTorch deploy params: {deploy_params:,})")

print("\n2. Copying weights from PyTorch -> TensorFlow...")
deploy_model.cpu().eval()
_np = lambda t: t.detach().cpu().numpy()
n_copied = 0
def cp_conv(kname, tconv):
    global n_copied
    tf_model.get_layer(kname).set_weights([_np(tconv.weight).transpose(2, 3, 1, 0)]); n_copied += 1
def cp_bn(kname, tbn):
    global n_copied
    tf_model.get_layer(kname).set_weights([_np(tbn.weight), _np(tbn.bias), _np(tbn.running_mean), _np(tbn.running_var)]); n_copied += 1

feat = deploy_model.features                       # [conv1, bn1, relu, maxpool, layer1..layer4]
cp_conv("stem_conv", feat[0]); cp_bn("stem_bn", feat[1])
for li, (n_blocks, mid, s) in enumerate(RESNET50_CFG, 1):
    layer = feat[3 + li]
    for bi, blk in enumerate(layer):
        nm = f"l{li}b{bi}"
        for k in (1, 2, 3):
            cp_conv(f"{nm}_conv{k}", getattr(blk, f"conv{k}")); cp_bn(f"{nm}_bn{k}", getattr(blk, f"bn{k}"))
        if blk.downsample is not None:
            cp_conv(f"{nm}_down_conv", blk.downsample[0]); cp_bn(f"{nm}_down_bn", blk.downsample[1])
fc = deploy_model.classifier[1]
tf_model.get_layer("cls_fc").set_weights([_np(fc.weight).T, _np(fc.bias)]); n_copied += 1
print(f"   Copied {n_copied} layer weight groups (expect 53 conv + 53 BN + 1 fc = 107)")

# ── Verify against PyTorch on target-TEST (one image at a time: the Keras input has a fixed batch of 1) ──
print("\n3. Verifying TF model accuracy on target test set...")
correct, total = 0, 0
for xs, xt, xw_s, xw_t, y, d in tgt_test_loader:
    raw01 = (xs * 0.5 + 0.5).numpy().transpose(0, 2, 3, 1)
    for i in range(len(y)):
        o = tf_model(raw01[i:i+1].astype(np.float32), training=False).numpy()
        correct += int(np.argmax(o[0]) == int(y[i])); total += 1
tf_acc = correct / total
print(f"   TF Keras model accuracy on target test: {tf_acc:.4f}")
print(f"   PyTorch deploy accuracy was:            {tgt_acc:.4f}")
print(f"   Difference: {abs(tf_acc - tgt_acc):.4f}")
if abs(tf_acc - tgt_acc) > 0.05:
    print("   ⚠️ Accuracy difference > 5% -- weight transfer is wrong, do NOT trust the TFLite numbers below")
else:
    print("   ✅ Accuracy within tolerance -- weight transfer successful!")

print("\n4. Converting to TFLite FP32...")
tfl_fp32 = tf.lite.TFLiteConverter.from_keras_model(tf_model).convert()
tflite_fp32_path = f"{MODELS_DIR}/resnet50_student_fp32.tflite"
with open(tflite_fp32_path, 'wb') as f:
    f.write(tfl_fp32)
print(f"   FP32 TFLite: {os.path.getsize(tflite_fp32_path)/1024:.1f} KB")

print("\n5. Converting to TFLite INT8 with representative dataset (300 calibration images)...")
def representative_dataset_gen():
    for i in range(min(300, cal_nhwc.shape[0])):
        yield [(cal_nhwc[i:i+1] * 0.5 + 0.5).astype(np.float32)]     # cal_nhwc is [-1,1]; model wants raw [0,1]

converter = tf.lite.TFLiteConverter.from_keras_model(tf_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_path = f"{MODELS_DIR}/resnet50_student_int8.tflite"
try:
    tfl_int8 = converter.convert()
    with open(tflite_int8_path, 'wb') as f:
        f.write(tfl_int8)
    fp32_size = os.path.getsize(tflite_fp32_path) / 1024
    int8_size = os.path.getsize(tflite_int8_path) / 1024
    print(f"   ✅ Pure full-INT8 conversion succeeded!  INT8 TFLite: {int8_size:.1f} KB")
    print(f"   FP32 -> INT8 compression: {fp32_size/int8_size:.1f}x")
except Exception as e:
    print(f"   ❌ INT8 conversion FAILED: {e}")
    import traceback; traceback.print_exc()
    tflite_int8_path = None

print(f"\nFP32 tflite: {tflite_fp32_path}")
print(f"INT8 tflite: {tflite_int8_path}")


1. Building TF/Keras ResNet-50...
   TF model params: 23,567,299   (PyTorch deploy params: 23,514,179)

2. Copying weights from PyTorch -> TensorFlow...
   Copied 107 layer weight groups (expect 53 conv + 53 BN + 1 fc = 107)

3. Verifying TF model accuracy on target test set...
   TF Keras model accuracy on target test: 0.8101
   PyTorch deploy accuracy was:            0.8101
   Difference: 0.0000
   ✅ Accuracy within tolerance -- weight transfer successful!

4. Converting to TFLite FP32...
INFO:tensorflow:Assets written to: /tmp/tmpo7xyjpyc/assets


INFO:tensorflow:Assets written to: /tmp/tmpo7xyjpyc/assets


Saved artifact at '/tmp/tmpo7xyjpyc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  133217874166608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874166416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874169488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133232810098128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874167760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874166800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874171408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874171792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874168336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874168720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874171600: TensorSpec

W0000 00:00:1790229088.349863      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790229088.349913      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1790229088.537943      23 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


   FP32 TFLite: 91777.6 KB

5. Converting to TFLite INT8 with representative dataset (300 calibration images)...
INFO:tensorflow:Assets written to: /tmp/tmpxi74v_j4/assets


INFO:tensorflow:Assets written to: /tmp/tmpxi74v_j4/assets


Saved artifact at '/tmp/tmpxi74v_j4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  133217874166608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874166416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874169488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133232810098128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874167760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874166800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874171408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874171792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874168336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874168720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133217874171600: TensorSpec

W0000 00:00:1790229103.475829      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1790229103.475867      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


   ✅ Pure full-INT8 conversion succeeded!  INT8 TFLite: 23671.6 KB
   FP32 -> INT8 compression: 3.9x

FP32 tflite: /kaggle/working/resnet50_student/models/resnet50_student_fp32.tflite
INT8 tflite: /kaggle/working/resnet50_student/models/resnet50_student_int8.tflite


In [ ]:
# ============================================================
# PART 10C: POST-QUANTIZATION ACCURACY + TFLITE MICRO OP CHECK
# ============================================================
quant = OrderedDict()
quant['PyTorch FP32'] = {'acc': tgt_acc, 'f1': tgt_f1, 'size_kb': deploy_params * 4 / 1024}

try:
    import tensorflow as tf

    def run_tflite(path, loader, is_int8):
        it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
        ind, outd = it.get_input_details()[0], it.get_output_details()[0]
        sc = ind['quantization_parameters']['scales']
        zp = ind['quantization_parameters']['zero_points']
        P, L = [], []
        for xs, xt, xw_s, xw_t, y, d in loader:
            raw01 = xs * 0.5 + 0.5   # model now expects RAW [0,1] pixels, not [-1,1]
            for i in range(len(xs)):
                a = np.transpose(raw01[i].unsqueeze(0).numpy().astype(np.float32), (0, 2, 3, 1))
                if is_int8 and len(sc) and sc[0] != 0:
                    a = np.clip(np.round(a / sc[0] + zp[0]), -128, 127).astype(ind['dtype'])
                it.set_tensor(ind['index'], a); it.invoke()
                P.append(int(np.argmax(it.get_tensor(outd['index'])[0]))); L.append(int(y[i]))
        return accuracy_score(L, P), f1_score(L, P, average='weighted')

    for nm_, pth, i8 in [('TFLite FP32', tflite_fp32_path, False),
                         ('TFLite INT8', tflite_int8_path, True)]:
        if pth and os.path.exists(pth):
            a, f = run_tflite(pth, tgt_test_loader, i8)
            quant[nm_] = {'acc': a, 'f1': f, 'size_kb': os.path.getsize(pth) / 1024}
            print(f"{nm_}: acc={a:.4f} f1={f:.4f}")

    if 'TFLite INT8' in quant:
        drop = tgt_acc - quant['TFLite INT8']['acc']
        print(f"\nAccuracy drop PyTorch -> INT8: {drop:+.4f}")
        print("  < 0.02 fine | 0.02-0.05 consider QAT | > 0.05 QAT or mixed precision needed")
except Exception as e:
    print("TFLite eval skipped:", e)

print(f"\n{'Format':<18}{'Acc':<10}{'F1':<10}{'Size KB':<12}{'Reduction':<10}")
print("-" * 62)
b = quant['PyTorch FP32']['size_kb']
for k, v in quant.items():
    print(f"{k:<18}{v['acc']:<10.4f}{v['f1']:<10.4f}{v['size_kb']:<12.1f}{b/v['size_kb']:<10.1f}x")
pd.DataFrame(quant).T.to_csv(f"{TABLES_DIR}/quantization.csv")

if tflite_int8_path and os.path.exists(tflite_int8_path):
    try:
        from tflite.Model import Model
        from tflite.BuiltinOperator import BuiltinOperator
        id2n = {v: k for k, v in BuiltinOperator.__dict__.items() if isinstance(v, int)}
        mdl = Model.GetRootAsModel(open(tflite_int8_path, 'rb').read(), 0)
        ops = []
        for i in range(mdl.OperatorCodesLength()):
            oc = mdl.OperatorCodes(i)
            cc = oc.CustomCode()
            ops.append(f"CUSTOM:{cc.decode()}" if cc else id2n.get(oc.BuiltinCode(), "UNKNOWN"))
        print("\nOps used:", ", ".join(sorted(set(ops))))
        flex = [o for o in ops if 'flex' in o.lower()]
        if flex:
            print("FLEX ops present -- these will NOT run on TFLite Micro:", set(flex))
        else:
            risky = set(ops) & {'BATCH_MATMUL', 'DIV', 'RSQRT', 'SQRT', 'TRANSPOSE', 'ELU',
                                'CUMSUM', 'SUM', 'MEAN'}
            print("No Flex ops. Verify these are registered in your ESP32 build:", risky or "none")
    except Exception as e:
        print("op check skipped:", e)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


TFLite FP32: acc=0.8101 f1=0.8108
TFLite INT8: acc=0.8101 f1=0.8108

Accuracy drop PyTorch -> INT8: +0.0000
  < 0.02 fine | 0.02-0.05 consider QAT | > 0.05 QAT or mixed precision needed

Format            Acc       F1        Size KB     Reduction 
--------------------------------------------------------------
PyTorch FP32      0.8101    0.8108    91852.3     1.0       x
TFLite FP32       0.8101    0.8108    91777.6     1.0       x
TFLite INT8       0.8101    0.8108    23671.6     3.9       x

Ops used: ADD, CONV_2D, FULLY_CONNECTED, MAX_POOL_2D, MEAN, MUL, PAD, SUB
No Flex ops. Verify these are registered in your ESP32 build: {'MEAN'}


In [ ]:
# ============================================================
# VIZ 10C-1: Quantization accuracy comparison
# ============================================================
if quant:
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(quant.keys())
    accs = [quant[n]['acc'] for n in names]
    colors = ['#1565C0', '#43A047', '#FF6F00'][:len(names)]
    bars = ax.bar(names, accs, color=colors, alpha=0.85)
    ax.set_ylabel('Accuracy')
    ax.set_title('Quantization — Accuracy Comparison', fontweight='bold')
    ax.set_ylim(0, 1.05)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                f'{b.get_height():.4f}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/quant_accuracy.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# VIZ 10C-2: Quantization model size comparison
# ============================================================
if quant:
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(quant.keys())
    sizes = [quant[n]['size_kb'] for n in names]
    colors = ['#1565C0', '#43A047', '#FF6F00'][:len(names)]
    bars = ax.bar(names, sizes, color=colors, alpha=0.85)
    ax.set_ylabel('Size (KB)')
    ax.set_title('Quantization — Model Size Comparison', fontweight='bold')
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+5,
                f'{b.get_height():.1f} KB', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/quant_size.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================
# PART 11: SUMMARY
# ============================================================
_active_da = '+'.join(n for n, u in [('CORAL', USE_CORAL), ('MMD', USE_MMD),
                                       ('Information Maximization', USE_IM),
                                       ('MCC', USE_MCC)] if u) or 'none'
summary = f"""
{'='*78}
  ResNet-50 student (same IM+pseudo+KD+BG-rand+AdaBN pipeline) -- Summary
{'='*78}
  Architecture   : torchvision ResNet-50 (ImageNet-init={STUDENT_PRETRAINED}), student replaces the custom MicroBrainNet
  Input          : {IMG_SIZE}x{IMG_SIZE}x3
  Adaptation     : {_active_da}  (CORAL/MMD/MCC disabled this run -- Part 0 switches)
                   + EMA/class-balanced pseudo-labels (auto-disabling)
                   + KD (logit + feature) from {TEACHER_BACKBONE}
                   + background randomization + AdaBN

  Params (deploy): {deploy_params:,}
  INT8 size      : {deploy_params/1024:.0f} KB  ({deploy_params/1024/500*100:.0f}% of the 500 KB budget)

  Source train (clean) : {src_train_acc:.4f}
  Source val           : {src_acc:.4f}
  Source test          : {src_test_acc:.4f}
  TARGET test          : {tgt_acc:.4f}   f1={tgt_f1:.4f}
  Bootstrap 95% CI     : [{lo:.4f}, {hi:.4f}]   (n={len(tgt_labels)})
  Teacher target test  : {t_te_acc if USE_TEACHER_KD else float('nan'):.4f}
  Generalization gap   : {src_train_acc - tgt_acc:.4f}

  Sanity gate (source-only, no DA/KD): src-val={sanity_src_val if sanity_src_val else float('nan'):.4f}
{'='*78}

  WHAT TO REPORT

  1. Target accuracy with the bootstrap CI, never the bare point estimate.
     79 test images -> 1 image = {100/len(tgt_labels):.2f} accuracy points.
  2. This is a SINGLE-seed (42) run; say so in the paper.
  3. Protocol: checkpoint selection, early stopping and the pseudo-label gate all read a
     labelled 79-image target validation split. This is semi-supervised DA, not pure UDA.
     State it -- a reviewer will find it otherwise.
  4. Frame the contribution as MCU-deployable cross-domain inference under a 500 KB INT8
     budget, with the ablation as evidence for each component. Published PlantVillage->PlantDoc
     work sits far lower on the full 27-class problem; this is 3 classes and not comparable.

  IF TARGET ACCURACY IS STILL SHORT, IN ORDER OF EXPECTED RETURN

  a) Teacher backbone: efficientnet_b0 -> convnext_tiny -> DINOv2 ViT-S frozen + linear probe.
     The student cannot exceed the teacher, so every point here propagates down.
  b) More target data. 364 training images is the binding constraint. Any additional field
     imagery (PlantWild, Plant-Pathology, your own phone photos) helps more than any loss term.
  c) Background randomization strength: raise BG_P to 0.7, expand BG_BANK beyond 180 crops.
  d) IMG_SIZE 160 with channels (32,48,64,96,128) -- still under 500 KB INT8.
  e) Only then: tune LAMBDA_IM, or re-enable CORAL/MMD/MCC (USE_CORAL/USE_MMD/USE_MCC in
     Part 0) and compare against this IM-only run's target accuracy above.
"""
print(summary)
with open(f"{OUTPUT_DIR}/summary.txt", "w") as f:
    f.write(summary)

print("Generated files:")
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, OUTPUT_DIR):<52s} {os.path.getsize(p)/1024:8.1f} KB")

import shutil
z = shutil.make_archive("/kaggle/working/resnet50_student_results", 'zip', OUTPUT_DIR)
print(f"\nZIP: {z} ({os.path.getsize(z)/1024/1024:.1f} MB)")


  ResNet-50 student (same IM+pseudo+KD+BG-rand+AdaBN pipeline) -- Summary
  Architecture   : torchvision ResNet-50 (ImageNet-init=True), student replaces the custom MicroBrainNet
  Input          : 224x224x3
  Adaptation     : Information Maximization  (CORAL/MMD/MCC disabled this run -- Part 0 switches)
                   + EMA/class-balanced pseudo-labels (auto-disabling)
                   + KD (logit + feature) from mobilenet_v3_large
                   + background randomization + AdaBN

  Params (deploy): 23,514,179
  INT8 size      : 22963 KB  (4593% of the 500 KB budget)

  Source train (clean) : 0.9989
  Source val           : 0.9982
  Source test          : 1.0000
  TARGET test          : 0.8101   f1=0.8108
  Bootstrap 95% CI     : [0.7215, 0.8861]   (n=79)
  Teacher target test  : 0.8228
  Generalization gap   : 0.1887

  Sanity gate (source-only, no DA/KD): src-val=1.0000

  WHAT TO REPORT

  1. Target accuracy with the bootstrap CI, never the bare point estimate.
     79 

In [ ]:
# ============================================================
# PART 12 (optional): export target-test images for Edge Impulse re-testing
# ============================================================
EXPORT_DIR = "/kaggle/working/ei_test_export"
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)
os.makedirs(EXPORT_DIR, exist_ok=True)

MAX_PER_CLASS = 40
rng = random.Random(SEED)
pairs = list(zip(tgt_test_idx.img_paths, tgt_test_idx.labels))
rng.shuffle(pairs)
cnt = {c: 0 for c in CLASSES}
for p, li in pairs:
    c = CLASSES[li]
    if cnt[c] >= MAX_PER_CLASS:
        continue
    d = os.path.join(EXPORT_DIR, c)
    os.makedirs(d, exist_ok=True)
    shutil.copy2(p, os.path.join(d, os.path.basename(p)))
    cnt[c] += 1

print("Exported (original files, no resize/normalize):")
for c, n in cnt.items():
    print(f"  {c}: {n}")
z2 = shutil.make_archive("/kaggle/working/ei_test_export", 'zip', EXPORT_DIR)
print(f"Zipped: {z2}")
print("Edge Impulse 'Model testing' expects per-class folder labels. Set its preprocessing to")
print(f"{IMG_SIZE}x{IMG_SIZE} RGB with [-1,1] scaling, or the numbers will not match this notebook.")

Exported (original files, no resize/normalize):
  Tomato_Septoria_Leaf_Spot: 24
  Potato_Early_Blight: 26
  Corn_Leaf_Blight: 29
Zipped: /kaggle/working/ei_test_export.zip
Edge Impulse 'Model testing' expects per-class folder labels. Set its preprocessing to
224x224 RGB with [-1,1] scaling, or the numbers will not match this notebook.
